In [4]:
import os

import pandas as pd
import numpy as np
import itertools
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.ticker import MultipleLocator
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.transforms import ScaledTranslation

plt.rcParams['legend.handlelength'] = 1
plt.rcParams['legend.handleheight'] = 1.125
plt.rcParams["font.family"]         = "Avenir"

path_to_scenarios = '/Users/Guille/Desktop/india_power/scenarios'
path_to_images    = '/Users/Guille/Desktop/india_power/images'
path_to_tables    = '/Users/Guille/Desktop/india_power/tables'
path_to_csvs      = '/Users/Guille/Desktop/india_power/gridpath_india_viz/csvs'
path_to_inputs    = '/Users/Guille/Desktop/india_power/input_data'

# Results

## Costs, emissions, and clean energy shares

### In the near-term (2030)

In [15]:
df_ = pd.read_csv(path_to_tables + '/pier-capacity.csv')
print(df_['scenario'].unique())

idx_ = ((df_['scenario'] == 'No Clean Target or Carbon Target') 
          & (df_['period'] == 2030))

df_ = df_.loc[idx_].reset_index(drop = True)
print(df_)
df_ = df_.groupby(['technology', 
                   'status']).agg({'capacity_mw': 'sum',
                                   'capacity_mwh': 'sum'}).reset_index(drop = False)

B  = df_.loc[df_['technology'] == 'Biomass', 'capacity_mw'].to_numpy()[0]
B += df_.loc[df_['technology'] == 'Hydro_ROR', 'capacity_mw'].to_numpy()[0]
B += df_.loc[df_['technology'] == 'Hydro_Storage', 'capacity_mw'].to_numpy()[0]
B += df_.loc[df_['technology'] == 'Nuclear', 'capacity_mw'].to_numpy()[0]
B += df_.loc[df_['technology'] == 'Offshore', 'capacity_mw'].to_numpy()[0]
B += df_.loc[df_['technology'] == 'SolarPV_roof', 'capacity_mw'].to_numpy()[0]
B += df_.loc[df_['technology'] == 'SolarPV_single', 'capacity_mw'].to_numpy()[0]
B += df_.loc[df_['technology'] == 'SolarPV_tilt', 'capacity_mw'].to_numpy()[0]
B += df_.loc[df_['technology'] == 'WHR', 'capacity_mw'].to_numpy()[0]
B += df_.loc[df_['technology'] == 'Wind', 'capacity_mw'].to_numpy()[0]
print(B)

['No Clean Target or Carbon Target' 'No Carbon Target' '80% Carbon Target'
 '90% Carbon Target - REF' '100% Carbon Target']
     period              technology           load_zone    status  \
0      2030                 Battery      Andhra_Pradesh       new   
1      2030                 Biomass      Andhra_Pradesh  existing   
2      2030                    CCGT      Andhra_Pradesh  existing   
3      2030                    CCGT      Andhra_Pradesh       new   
4      2030                      CT      Andhra_Pradesh       new   
..      ...                     ...                 ...       ...   
389    2030          SolarPV_single  Dadra_Nagar_Haveli       new   
390    2030            SolarPV_tilt  Dadra_Nagar_Haveli  existing   
391    2030            SolarPV_tilt  Dadra_Nagar_Haveli       new   
392    2030  Subcritical_Coal_Small  Dadra_Nagar_Haveli  existing   
393    2030                    Wind  Dadra_Nagar_Haveli       new   

     capacity_mw  capacity_mwh                 

In [4]:
df_ = pd.read_csv(path_to_tables + '/pier-clean_energy.csv')
print(df_['scenario'].unique())

idx_1_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid') 
          & (df_['period'] == 2030)
          & (df_['technology'] == 'clean'))

idx_2_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid')
          & (df_['period'] == 2030)
          & (df_['technology'] == 'no_clean'))
# print(df_.loc[idx_1_])

print(df_.loc[idx_1_, 'power_mw'].to_numpy()[0])
print(df_.loc[idx_1_, 'power_mw'].to_numpy()[0]/( df_.loc[idx_1_, 'power_mw'].to_numpy()[0]+ df_.loc[idx_2_, 'power_mw'].to_numpy()[0]))

idx_1_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid') 
          & (df_['period'] == 2030)
          & (df_['technology'] == 'clean'))

idx_2_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid')
          & (df_['period'] == 2030)
          & (df_['technology'] == 'no_clean'))
# print(df_.loc[idx_1_])
# print(df_.loc[idx_2_])

print(df_.loc[idx_1_, 'power_mw'].to_numpy()[0]/( df_.loc[idx_1_, 'power_mw'].to_numpy()[0]+ df_.loc[idx_2_, 'power_mw'].to_numpy()[0]))

['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_80CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_100CAP_500GW_PIERmid']
   period                                           scenario technology  \
2    2030  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid      clean   

  load_zone      power_mw  
2       all  1.815916e+09  
1815915683.747472
0.6888850431105996
0.4528016852942397


In [407]:
df_ = pd.read_csv(path_to_tables + '/pier-system_cost.csv')

df_['system_cost'] = df_['variable_cost'] + df_['fixed_cost'] + 5.9e9
idx_1_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid') 
          & (df_['period'] == 2030))
print(df_.loc[idx_1_])

idx_2_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid')
          & (df_['period'] == 2030))
print(df_.loc[idx_2_])

print(1. - df_.loc[idx_1_, 'system_cost'].to_numpy()[0]/df_.loc[idx_2_, 'system_cost'].to_numpy()[0])

   period                                           scenario load_zone  \
1    2030  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid       all   

   variable_cost    fixed_cost   system_cost  
1   2.456985e+10  9.756039e+10  1.280302e+11  
    period                                           scenario load_zone  \
13    2030  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...       all   

    variable_cost    fixed_cost   system_cost  
13   4.677061e+10  8.471879e+10  1.373894e+11  
0.06812142567255086


In [336]:
df_ = pd.read_csv(path_to_tables + '/pier-emissions.csv')
print(df_['scenario'].unique())

idx_1_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid') 
          & (df_['period'] == 2020))
idx_2_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid')
          & (df_['period'] == 2030))
# print(df_.loc[idx_1_])
# print(df_.loc[idx_2_])

print(df_.loc[idx_2_, 'carbon_emissions_tons'].to_numpy()[0]/df_.loc[idx_1_, 'carbon_emissions_tons'].to_numpy()[0])

idx_1_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid') 
          & (df_['period'] == 2020))
idx_2_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid')
          & (df_['period'] == 2030))
# print(df_.loc[idx_1_])
# print(df_.loc[idx_2_])

print(df_.loc[idx_2_, 'carbon_emissions_tons'].to_numpy()[0]/df_.loc[idx_1_, 'carbon_emissions_tons'].to_numpy()[0])

['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_80CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_100CAP_500GW_PIERmid']
0.6035177647178793
1.1275201252528861


### In the medium term (2040)

In [22]:
df_1_ = pd.read_csv(path_to_tables + '/pier-system_cost.csv')
df_1_['total_cost'] = df_1_['variable_cost'] + df_1_['fixed_cost'] + 5.9e9

idx_1_ = ((df_1_['period'] == 2020) & (df_1_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid'))
idx_2_ = ((df_1_['period'] == 2020) & (df_1_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'))
print(df_1_.loc[idx_1_, 'total_cost'].to_numpy()[0], df_1_.loc[idx_1_, 'total_cost'].to_numpy()[0])

idx_1_ = ((df_1_['period'] == 2040) & (df_1_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid'))
idx_2_ = ((df_1_['period'] == 2040) & (df_1_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'))
print(df_1_.loc[idx_1_, 'total_cost'].to_numpy()[0], df_1_.loc[idx_2_, 'total_cost'].to_numpy()[0])

df_2_ = pd.read_csv(path_to_tables + '/pier-emissions.csv')
print(df_2_)

idx_1_ = ((df_2_['period'] == 2020) & (df_2_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid'))
idx_2_ = ((df_2_['period'] == 2020) & (df_2_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'))

print(df_2_.loc[idx_1_, 'carbon_emissions_tons'].to_numpy()[0])
print(df_2_.loc[idx_2_, 'carbon_emissions_tons'].to_numpy()[0])

    period                                           scenario load_zone  \
0     2020  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid       all   
1     2030  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid       all   
2     2040  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid       all   
3     2050  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid       all   
4     2020  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500G...       all   
5     2030  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500G...       all   
6     2040  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500G...       all   
7     2050  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500G...       all   
8     2020  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_80CA...       all   
9     2030  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_80CA...       all   
10    2040  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_80CA...       all   
11    2050  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_80CA...       all   
12    2020  VREmid_STmid_

687000000
    period                                           scenario load_zone  \
0     2020  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid       all   
1     2030  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid       all   
2     2040  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid       all   
3     2050  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid       all   
4     2020  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500G...       all   
5     2030  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500G...       all   
6     2040  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500G...       all   
7     2050  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500G...       all   
8     2020  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_80CA...       all   
9     2030  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_80CA...       all   
10    2040  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_80CA...       all   
11    2050  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_80CA...       all   
12    2020  VRE

In [350]:
df_ = pd.read_csv(path_to_tables + '/pier-emissions.csv')
print(df_['scenario'].unique())

df_['intensity'] = df_['carbon_emissions_tons']/df_['load_mw']

idx_1_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid') 
          & (df_['period'] == 2020))

idx_2_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid')
          & (df_['period'] == 2040))
# print(df_.loc[idx_1_])
# print(df_.loc[idx_2_])

print(df_.loc[idx_1_, 'intensity'].to_numpy()[0], df_.loc[idx_2_, 'intensity'].to_numpy()[0])
print(df_.loc[idx_1_, 'intensity'].to_numpy()[0] - df_.loc[idx_2_, 'intensity'].to_numpy()[0])
print(1. - df_.loc[idx_2_, 'intensity'].to_numpy()[0]/df_.loc[idx_1_, 'intensity'].to_numpy()[0])

['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_80CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_100CAP_500GW_PIERmid']
0.7288078360490977 0.21257079491309525
0.5162370411360024
0.7083308049136083


In [25]:
df_1_ = pd.read_csv(path_to_tables + '/pier-system_cost.csv')
df_2_ = pd.read_csv(path_to_tables + '/pier-emissions.csv')
df_   = pd.merge(df_1_, df_2_, on=['period', 'scenario'], how='inner')

df_['levelized'] = (df_['variable_cost'] + df_['fixed_cost'] + 5.9e9)/df_['load_mw']

idx_1_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid') 
          & (df_['period'] == 2020))
print(df_.loc[idx_1_])

idx_2_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid')
          & (df_['period'] == 2040))
print(df_.loc[idx_2_])

print(df_.loc[idx_1_, 'levelized'].to_numpy()[0], df_.loc[idx_2_, 'levelized'].to_numpy()[0])
print(df_.loc[idx_1_, 'levelized'].to_numpy()[0]*72/1000, df_.loc[idx_2_, 'levelized'].to_numpy()[0]*72/1000)

print(df_.loc[idx_1_, 'levelized'].to_numpy()[0] - df_.loc[idx_2_, 'levelized'].to_numpy()[0])
print(1. - df_.loc[idx_2_, 'levelized'].to_numpy()[0]/df_.loc[idx_1_, 'levelized'].to_numpy()[0])

   period                                           scenario load_zone_x  \
4    2020  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500G...         all   

   variable_cost    fixed_cost load_zone_y  carbon_emissions_tons  \
4   3.944529e+10  4.207209e+10         all           1.119110e+09   

        load_mw  levelized  
4  1.535536e+09  56.929569  
   period                                           scenario load_zone_x  \
6    2040  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500G...         all   

   variable_cost    fixed_cost load_zone_y  carbon_emissions_tons  \
6   3.435344e+10  1.493140e+11         all           8.904444e+08   

        load_mw  levelized  
6  4.188931e+09  45.254377  
56.92956897594688 45.254377320191864
4.098928966268175 3.258315167053814
11.675191655755015
0.20508132883788144


In [29]:
df_1_ = pd.read_csv(path_to_inputs + '/policy/CAP_targets_v9.csv')
print(df_1_.loc[df_1_['period'] == 2040., '90CAP_2040_2050'].to_numpy()[0])

df_2_ = pd.read_csv(path_to_tables + '/pier-emissions.csv')

idx_1_ = ((df_2_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid') 
          & (df_2_['period'] == 2020))
print(df_2_.loc[idx_1_])

print(df_1_.loc[df_1_['period'] == 2040., '90CAP_2040_2050'].to_numpy()[0]/df_2_.loc[idx_1_, 'carbon_emissions_tons'].to_numpy()[0])

687000000
   period                                           scenario load_zone  \
4    2020  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500G...       all   

   carbon_emissions_tons       load_mw  
4           1.119110e+09  1.535536e+09  
0.6138804265699843


### In the long-term (2050)

In [409]:
df_1_ = pd.read_csv(path_to_tables + '/pier-system_cost.csv')
df_2_ = pd.read_csv(path_to_tables + '/pier-emissions.csv')
df_   = pd.merge(df_1_, df_2_, on=['period', 'scenario'], how='inner')
print(df_['scenario'].unique())

df_['levelized'] = (df_['variable_cost'] + df_['fixed_cost'] + 5.9e9)/df_['load_mw']

idx_1_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid') 
          & (df_['period'] == 2050))
#print(df_.loc[idx_1_])

idx_2_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_80CAP_500GW_PIERmid')
          & (df_['period'] == 2050))
#print(df_.loc[idx_2_])

idx_3_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid')
          & (df_['period'] == 2050))
#print(df_.loc[idx_3_])

idx_4_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_100CAP_500GW_PIERmid')
          & (df_['period'] == 2050))
#print(df_.loc[idx_4_])

print(df_.loc[idx_2_, 'levelized'].to_numpy()[0]/df_.loc[idx_1_, 'levelized'].to_numpy()[0] - 1)
print(df_.loc[idx_3_, 'levelized'].to_numpy()[0]/df_.loc[idx_1_, 'levelized'].to_numpy()[0] - 1)
print(df_.loc[idx_4_, 'levelized'].to_numpy()[0]/df_.loc[idx_1_, 'levelized'].to_numpy()[0] - 1)

['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_80CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_100CAP_500GW_PIERmid']
0.07196721301843945
0.10898881195367438
0.21503372031524814


In [365]:
df_ = pd.read_csv(path_to_tables + '/pier-emissions.csv')
print(df_['scenario'].unique())

idx_1_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid') 
          & (df_['period'] == 2020))

idx_2_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid')
          & (df_['period'] == 2050))
# print(df_.loc[idx_1_])
# print(df_.loc[idx_2_])

print(df_.loc[idx_1_, 'carbon_emissions_tons'].to_numpy()[0], df_.loc[idx_2_, 'carbon_emissions_tons'].to_numpy()[0])
print(1. - df_.loc[idx_2_, 'carbon_emissions_tons'].to_numpy()[0]/df_.loc[idx_1_, 'carbon_emissions_tons'].to_numpy()[0])

['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_80CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_100CAP_500GW_PIERmid']
1119110449.3078015 938375704.7215072
0.16149857656863398


In [218]:
df_1_ = pd.read_csv(path_to_tables + '/pier-emissions.csv')
df_2_ = pd.read_csv(path_to_tables + '/pier-system_cost.csv')

print(df_1_['scenario'].unique())
print(df_2_['scenario'].unique())

idx_1_ = df_1_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
idx_2_ = df_2_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'

df_ = pd.merge(df_1_.loc[idx_1_], df_2_.loc[idx_2_], on = ['period', 'scenario', 'load_zone'], how = 'right')

df_['intensity']  = df_['carbon_emissions_tons']/df_['load_mw']
df_['total_cost'] = df_['variable_cost'] + df_['fixed_cost'] + 5.9e9
df_['lcoe'] = df_['total_cost']/df_['load_mw']
df_['lcoe_ind'] = df_['lcoe']*72/1000

print(df_)

print(1. - df_.loc[df_['period'] == 2050, 'intensity'].to_numpy()[0]/df_.loc[df_['period'] == 2020, 'intensity'].to_numpy()[0])
print(1. - df_.loc[df_['period'] == 2050, 'lcoe'].to_numpy()[0]/df_.loc[df_['period'] == 2020, 'lcoe'].to_numpy()[0])

['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_80CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_100CAP_500GW_PIERmid']
['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_80CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_100CAP_500GW_PIERmid']
   period                                           scenario load_zone  \
0    2020  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...       all   
1    2030  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...       all   
2    2040  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...       all   
3    2050  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...       all   

   carbon_e

### VRE and storage technology costs impact

In [394]:
df_ = pd.read_csv(path_to_tables + '/cost-system_cost.csv')

df_['system_cost']= df_['variable_cost'] + df_['fixed_cost'] + 5.9e9

df_['min_system_cost'] = df_['system_cost']
df_['max_system_cost'] = df_['system_cost']

df_ = df_.groupby(['period']).agg({'min_system_cost': 'min',
                                   'max_system_cost': 'max'}).reset_index(drop = False)

df_['max_inc_system_cost'] = (df_['max_system_cost'] - df_['max_system_cost'].to_numpy()[0])/1e9
df_['min_inc_system_cost'] = (df_['min_system_cost'] - df_['min_system_cost'].to_numpy()[0])/1e9

df_['max_inc_system_cost_cr'] = 72*(df_['max_system_cost'] - df_['max_system_cost'].to_numpy()[0])/1e7
df_['min_inc_system_cost_cr'] = 72*(df_['min_system_cost'] - df_['min_system_cost'].to_numpy()[0])/1e7

df_['max_times_system_cost'] = df_['max_system_cost']/df_['max_system_cost'].to_numpy()[0]
df_['min_times_system_cost'] = df_['min_system_cost']/df_['min_system_cost'].to_numpy()[0]
print(df_)

   period  min_system_cost  max_system_cost  max_inc_system_cost  \
0    2020     8.151739e+10     8.151739e+10             0.000000   
1    2030     1.246794e+11     1.387361e+11            57.218720   
2    2040     1.599743e+11     2.068841e+11           125.366703   
3    2050     2.087623e+11     3.165819e+11           235.064466   

   min_inc_system_cost  max_inc_system_cost_cr  min_inc_system_cost_cr  \
0             0.000000            0.000000e+00                0.000000   
1            43.162040            4.119748e+05           310766.684936   
2            78.456922            9.026403e+05           564889.841075   
3           127.244922            1.692464e+06           916163.437450   

   max_times_system_cost  min_times_system_cost  
0               1.000000               1.000000  
1               1.701920               1.529483  
2               2.537914               1.962456  
3               3.883611               2.560954  


In [403]:
df_1_ = pd.read_csv(path_to_tables + '/cost-system_cost.csv')
df_2_ = pd.read_csv(path_to_tables + '/cost-emissions.csv')
df_   = pd.merge(df_1_, df_2_, on = ['period', 'scenario'], how = 'inner')

df_['system_cost'] = df_['variable_cost'] + df_['fixed_cost'] + 5.9e9
df_['levelized']   = df_['system_cost'] / df_['load_mw']

idx_1_ = df_['period'] == 2020
idx_2_ = df_['period'] == 2050

print(df_.loc[idx_1_])
print(df_.loc[idx_2_])

print(df_.loc[idx_2_, 'levelized'].to_numpy()/df_.loc[idx_1_, 'levelized'].to_numpy())

    period                                           scenario load_zone_x  \
0     2020  VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...         all   
4     2020  VRElow_STlow_CONVhigh_H2_RES_8PRM_CC_50RPS_90C...         all   
8     2020  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...         all   
12    2020  VREmid_STmid_CONVhigh_H2_RES_8PRM_CC_50RPS_90C...         all   
16    2020  VREhigh_SThigh_CONVmid_H2_RES_8PRM_CC_50RPS_90...         all   
20    2020  VREhigh_SThigh_CONVhigh_H2_RES_8PRM_CC_50RPS_9...         all   

    variable_cost    fixed_cost load_zone_y  carbon_emissions_tons  \
0    3.944529e+10  4.207209e+10         all           1.119110e+09   
4    3.944529e+10  4.207209e+10         all           1.119110e+09   
8    3.944529e+10  4.207209e+10         all           1.119110e+09   
12   3.944529e+10  4.207209e+10         all           1.119110e+09   
16   3.944529e+10  4.207209e+10         all           1.119110e+09   
20   3.944529e+10  4.207209e+10         

### Demand projections

In [161]:
df_ = pd.read_csv(path_to_tables + '/demand-system_cost.csv')
print(df_['scenario'].unique())

period = 2050

df_['system_cost'] = df_['variable_cost'] + df_['fixed_cost'] + 5.9e9

idx_ = ((df_['period'] == period) 
        & (df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_ICEDlow'))

ICEDlow = df_.loc[idx_, 'system_cost'].to_numpy()[0]

idx_ = ((df_['period'] == period) 
        & (df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_ICEDmid'))

ICEDmid = df_.loc[idx_, 'system_cost'].to_numpy()[0]

idx_ = ((df_['period'] == period) 
        & (df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_ICEDhigh'))

ICEDhigh = df_.loc[idx_, 'system_cost'].to_numpy()[0]
print(ICEDlow, ICEDmid, ICEDhigh)

idx_ = ((df_['period'] == period) 
        & (df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERlow'))

PIERlow = df_.loc[idx_, 'system_cost'].to_numpy()[0]

idx_ = ((df_['period'] == period) 
        & (df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'))

PIERmid = df_.loc[idx_, 'system_cost'].to_numpy()[0]

idx_ = ((df_['period'] == period) 
        & (df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERhigh'))

PIERhigh = df_.loc[idx_, 'system_cost'].to_numpy()[0]
print(PIERlow, PIERmid, PIERhigh)

print(ICEDlow/PIERlow - 1, ICEDmid/PIERmid - 1, ICEDhigh/PIERhigh - 1)

['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_ICEDlow'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_ICEDmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_ICEDhigh'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERlow'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERhigh']
286300929653.05237 301566161107.6485 325666817284.57434
255893170766.38553 273397303042.45633 297627321233.94824
0.11882989606794636 0.10303268449146974 0.09421008775127149


## Cost-optimal capacity deployment and energy generation

In [36]:
path = '/Users/Guille/Desktop/india_power/data/gem/'
df_  = pd.read_excel(path + 'Global-Coal-Plant-Tracker-July-2025.xlsx', sheet_name = 'Units')
print(df_.shape)

df_ = df_.loc[df_['Country/Area'] == 'India'].reset_index(drop = True)
print(df_.shape)

df_ = df_.groupby(['Status']).agg({'Capacity (MW)': 'sum'}).reset_index(drop = False)
print(df_)

B  = df_.loc[df_['Status'] == 'announced', 'Capacity (MW)'].to_numpy()[0]
B += df_.loc[df_['Status'] == 'pre-permit', 'Capacity (MW)'].to_numpy()[0]
B += df_.loc[df_['Status'] == 'permitted', 'Capacity (MW)'].to_numpy()[0]
B += df_.loc[df_['Status'] == 'construction', 'Capacity (MW)'].to_numpy()[0]
print(B)

#          Status  Capacity (MW)
# 0     announced        32630.0
# 6    pre-permit        30430.0
# 5     permitted        29120.0
# 2  construction        27570.0

(14357, 47)
(1963, 47)
         Status  Capacity (MW)
0     announced        32630.0
1     cancelled       586074.0
2  construction        27570.0
3    mothballed          781.5
4     operating       245082.7
5     permitted        29120.0
6    pre-permit        30430.0
7       retired        17141.1
8       shelved         2615.0
119750.0


In [35]:
df_ = pd.read_csv(path_to_inputs + '/pipeline_generators_2025.csv')

df_ = df_.loc[df_['technology'] == 'Supercritical_Coal', 'gen_capacity'].sum()
print(df_)

36515.0


In [68]:
df_1_ = pd.read_csv(path_to_inputs + '/generators/existing_generators_2025.csv')
df_2_ = pd.read_csv(path_to_inputs + '/generators/pipeline_generators_2024.csv')
df_3_ = pd.read_csv(path_to_inputs + '/generators/existing_captive_generators_2022.csv')

idx_ = ((df_1_['technology'] == 'Subcritical_Coal_Small')
        | (df_1_['technology'] == 'Subcritical_Coal_Large')
        | (df_1_['technology'] == 'Supercritical_Coal'))

df_1_ = df_1_.loc[idx_].reset_index(drop = True)
df_1_ = df_1_[['status', 'technology', 'gen_capacity', 'year_of_commission']]

df_2_ = df_2_.loc[df_2_['technology'] == 'Supercritical_Coal'].reset_index(drop = True)
df_2_ = df_2_[['status', 'technology', 'gen_capacity', 'year_of_commission']]

df_3_ = df_3_.loc[idx_].reset_index(drop = True)
df_3_ = df_3_[['status', 'technology', 'gen_capacity', 'year_of_commission']]
df_3_['status'] = 'captive'

df_ = pd.concat([df_1_, df_2_, df_3_], axis = 0).reset_index(drop = True)
print(df_['gen_capacity'].sum())
print(df_.loc[df_['status'] == 'pipeline', 'gen_capacity'].sum())
print(df_.loc[df_['status'] == 'existing', 'gen_capacity'].sum())

for period in [2020, 2030, 2040, 2050]:
    df_['operation_years'] = period - df_['year_of_commission']

    idx_1_ = (df_['operation_years'] < 0)
    idx_2_ = (df_['operation_years'] > 45)

    print(period, df_.loc[~(idx_1_ | idx_2_ ), 'gen_capacity'].sum(), df_.loc[idx_2_, 'gen_capacity'].sum())
    print(period, df_.loc[~(idx_1_ | idx_2_ ), 'operation_years'].mean(), df_.loc[~(idx_1_ | idx_2_ ), 'operation_years'].median())

324263.52
73030.0
203474.0
2020 251051.02000000002 182.5
2020 14.823723228995057 9.0
2030 314400.02 9863.5
2030 21.476744186046513 18.0
2040 290863.02 33400.5
2040 27.913214990138066 27.0
2050 274225.52 50038.0
2050 35.929545454545455 36.0


In [73]:
df_ = pd.read_csv(path_to_tables + '/cost-grouped_capacity.csv')

df_p_ = df_.loc[((df_['technology'] == 'Coal')
                & (df_['period'] == 2050)
                & (df_['status'] == 'new'))].reset_index(drop = True)

df_p_ = df_p_.groupby(['scenario']).agg({'capacity_mw': 'sum'}).reset_index(drop = False)
#print(df_p_)
print(df_p_['capacity_mw'])

df_p_ = df_.loc[((df_['technology'] == 'Gas')
                & (df_['period'] == 2050)
                & (df_['status'] == 'new'))].reset_index(drop = True)
df_p_ = df_p_.groupby(['scenario']).agg({'capacity_mw': 'sum'}).reset_index(drop = False)
#print(df_p_)
print(df_p_)

0    36515.0
1    36515.0
2    36515.0
3    36515.0
4    36515.0
5    36515.0
Name: capacity_mw, dtype: float64
                           scenario  capacity_mw
0      VRE & ESS (high) Coal (high)  4499.250941
1       VRE & ESS (high) Coal (low)  4499.250941
2       VRE & ESS (low) Coal (high)     0.000000
3        VRE & ESS (low) Coal (low)     0.000000
4       VRE & ESS (mid) Coal (high)     0.000000
5  VRE & ESS (mid) Coal (low) - REF     0.000000


In [75]:
# I am splitting the results for PIER and ICED. 
# Do you see any new endogenously built coal capacity in any of the 3 PIER scenarios? 
# How much endogenously built coal capacity do you see in ICED scenarios? 
# How much endogenously built coal capacity do you see in high VRE and ESS costs and low coal costs? None, only 4.5GW of Gas 
df_ = pd.read_csv(path_to_tables + '/demand-grouped_capacity.csv')

df_p_ = df_.loc[((df_['technology'] == 'Coal')
                & (df_['period'] == 2050)
                & (df_['status'] == 'new'))].reset_index(drop = True)

df_p_ = df_p_.groupby(['scenario']).agg({'capacity_mw': 'sum'}).reset_index(drop = False)
#print(df_p_)
print(df_p_)

df_p_ = df_.loc[((df_['technology'] == 'Gas')
                & (df_['period'] == 2050)
                & (df_['status'] == 'new'))].reset_index(drop = True)
df_p_ = df_p_.groupby(['scenario']).agg({'capacity_mw': 'sum'}).reset_index(drop = False)
#print(df_p_)
print(df_p_)

                        scenario   capacity_mw
0        Bottom-up Demand (high)  42190.389643
1         Bottom-up Demand (low)  36515.000000
2   Bottom-up Demand (mid) - REF  36515.000000
3  Linearly-scaled Demand (high)  39238.089059
4   Linearly-scaled Demand (low)  36515.000000
5   Linearly-scaled Demand (mid)  36515.000000
                        scenario   capacity_mw
0        Bottom-up Demand (high)   3136.599743
1         Bottom-up Demand (low)      0.000000
2   Bottom-up Demand (mid) - REF      0.000000
3  Linearly-scaled Demand (high)  85313.697268
4   Linearly-scaled Demand (low)  67999.900486
5   Linearly-scaled Demand (mid)  72312.000427


### Coal and gas capacity

In [27]:
df_ = pd.read_csv(path_to_inputs + '/india_gen_2019-20.csv')
print(df_)

df_ = df_.groupby(['source']).agg({'generation_gwh': 'sum'})
print(df_)

fossil_fuels_frac = (df_.loc['Coal', 'generation_gwh'] + df_.loc['Oil & Gas', 'generation_gwh'])/df_['generation_gwh'].sum()
print(100*fossil_fuels_frac)

print(df_.loc['Coal', 'generation_gwh'])
print(100*df_.loc['Coal', 'generation_gwh']/df_['generation_gwh'].sum())

              state    FY       source power_mw  generation_gwh
0    Andhra Pradesh  2020         Coal    11590        56916.07
1    Andhra Pradesh  2020    Oil & Gas  4935.34         3013.62
2    Andhra Pradesh  2020        Hydro     1610         3013.43
3    Andhra Pradesh  2020        Solar  3698.32         5855.11
4    Andhra Pradesh  2020         Wind  4092.45         7626.64
..              ...   ...          ...      ...             ...
159     West Bengal  2020        Hydro   1341.2         2916.38
160     West Bengal  2020        Solar   127.24           64.29
161     West Bengal  2020  Small Hydro     98.5          115.68
162     West Bengal  2020    Bio Power   321.09         1295.07
163          Bhutan  2020        Hydro     2326         9186.00

[164 rows x 5 columns]
             generation_gwh
source                     
Bio Power          14107.06
Coal              993879.10
Hydro             164684.84
Nuclear            46472.45
Oil & Gas          48550.87
Small Hydro 

In [25]:
df_ = pd.read_csv(path_to_tables + '/input_data-electricity_system_groups.csv')
print(df_)

df_ = df_.groupby(['technology']).agg({'capacity_mw': 'sum'})
print(df_)

print(df_.loc['Coal', 'capacity_mw'])
print(df_.loc['Coal', 'capacity_mw']/df_['capacity_mw'].sum())

    technology       load_zone  capacity_mw
0         Coal  Andhra_Pradesh    11678.618
1         Coal           Assam      883.310
2         Coal           Bihar     5875.500
3         Coal      Chandigarh       14.400
4         Coal    Chhattisgarh    29179.048
..         ...             ...          ...
129       Wind  Madhya_Pradesh      503.000
130       Wind     Maharashtra     1000.000
131       Wind       Rajasthan      859.000
132       Wind      Tamil_Nadu     1860.000
133       Wind       Telangana       25.000

[134 rows x 3 columns]
                capacity_mw
technology                 
Coal             235403.520
Diesel             1213.960
Gas               24505.359
Hydro             42940.120
Nuclear            7880.000
Other             10300.837
Pumped Storage     3140.000
Solar              6824.000
Wind               7543.000
235403.52000000002
0.6928711360546747


In [52]:
df_ = pd.read_csv(path_to_tables + '/cost-grouped_dispatch.csv')

idx_ = ((df_['technology'] == 'Battery') 
        & (df_['technology'] == 'Transmission Losses')
        & (df_['technology'] == 'Pumped Storage')
        & (df_['technology'] == 'Shedding')
        & (df_['technology'] == 'Curtailment')
        & (df_['technology'] == 'Hydrogen'))

dis_p_ = df_.loc[~idx_].groupby(['period', 'scenario']).agg({'power_mw': 'sum'}).reset_index(drop = False)
dis_p_ = dis_p_.loc[(dis_p_['scenario'] == 'VRE & ESS (mid) Coal (low) - REF')]
print(dis_p_)

idx_ = (df_['technology'] == 'Coal') & (df_['scenario'] == 'VRE & ESS (mid) Coal (low) - REF')
dis_pp_ = df_.loc[idx_].groupby(['period', 'scenario']).agg({'power_mw': 'sum'}).reset_index(drop = False)
print(dis_pp_)

print(100*dis_pp_['power_mw'].to_numpy()/dis_p_['power_mw'].to_numpy())

    period                          scenario      power_mw
5     2020  VRE & ESS (mid) Coal (low) - REF  1.535536e+09
11    2030  VRE & ESS (mid) Coal (low) - REF  2.582122e+09
17    2040  VRE & ESS (mid) Coal (low) - REF  4.188931e+09
23    2050  VRE & ESS (mid) Coal (low) - REF  5.622802e+09
   period                          scenario      power_mw
0    2020  VRE & ESS (mid) Coal (low) - REF  1.124794e+09
1    2030  VRE & ESS (mid) Coal (low) - REF  1.278846e+09
2    2040  VRE & ESS (mid) Coal (low) - REF  9.020296e+08
3    2050  VRE & ESS (mid) Coal (low) - REF  1.034512e+08
[73.25093473 49.52694358 21.53364491  1.83985227]


In [96]:
df_1_ = pd.read_csv(path_to_tables + '/cost-grouped_capacity.csv')
df_2_ = pd.read_csv(path_to_tables + '/pier-grouped_capacity.csv')
df_3_ = pd.read_csv(path_to_tables + '/iced-grouped_capacity.csv')
df_4_ = pd.read_csv(path_to_tables + '/demand-grouped_capacity.csv')
df_5_ = pd.read_csv(path_to_tables + '/alternative-grouped_capacity.csv')
df_6_ = pd.read_csv(path_to_tables + '/additional-grouped_capacity.csv')

df_1_['group'] = 'cost'
df_2_['group'] = 'pier'
df_3_['group'] = 'iced'
df_4_['group'] = 'demand'
df_5_['group'] = 'alternative'
df_6_['group'] = 'additional'

df_ = pd.concat([df_1_, df_2_, df_3_, df_4_, df_5_, df_6_], axis = 0).reset_index(drop = True)

df_p_ = df_.loc[((df_['technology'] == 'Coal')
                & (df_['period'] == 2050)
                & (df_['status'] == 'new'))].reset_index(drop = True)
df_p_ = df_p_.groupby(['group', 'scenario']).agg({'capacity_mw': 'sum'}).reset_index(drop = False)
#print(df_p_)
print(df_p_['capacity_mw'].max())

df_p_ = df_.loc[((df_['technology'] == 'Gas')
                & (df_['period'] == 2050)
                & (df_['status'] == 'new'))].reset_index(drop = True)
df_p_ = df_p_.groupby(['group', 'scenario']).agg({'capacity_mw': 'sum'}).reset_index(drop = False)
#print(df_p_)
print(df_p_['capacity_mw'].max())

97945.75549961705
169867.60610229056


### Solar and wind capacity

In [76]:
df_ = pd.read_csv(path_to_tables + '/cost-grouped_capacity.csv')
print(df_['scenario'].unique())

df_ = df_.loc[(df_['technology'] == 'Solar') | (df_['technology'] == 'Wind')].reset_index(drop = True)
df_ = df_.groupby(['period', 'scenario']).agg({'capacity_mw': 'sum'})#.reset_index(drop = False)

print(df_.loc[2020])
print(df_.loc[2020, 'capacity_mw'].min(), df_.loc[2020, 'capacity_mw'].max())

print(df_.loc[2030])
print(df_.loc[2030, 'capacity_mw'].min(), df_.loc[2030, 'capacity_mw'].max())
print(df_.loc[2030, 'capacity_mw'].max()/df_.loc[2020, 'capacity_mw'].max())
print(df_.loc[2040])
print(df_.loc[2040, 'capacity_mw'].min(), df_.loc[2040, 'capacity_mw'].max())
print(df_.loc[2040, 'capacity_mw'].max()/df_.loc[2030, 'capacity_mw'].max())

print(df_.loc[2050])
print(df_.loc[2050, 'capacity_mw'].min(), df_.loc[2050, 'capacity_mw'].max())
print(df_.loc[2050, 'capacity_mw'].max()/df_.loc[2040, 'capacity_mw'].max())

['VRE & ESS (high) Coal (high)' 'VRE & ESS (high) Coal (low)'
 'VRE & ESS (low) Coal (high)' 'VRE & ESS (low) Coal (low)'
 'VRE & ESS (mid) Coal (high)' 'VRE & ESS (mid) Coal (low) - REF']
                                  capacity_mw
scenario                                     
VRE & ESS (high) Coal (high)          71678.0
VRE & ESS (high) Coal (low)           71678.0
VRE & ESS (low) Coal (high)           71678.0
VRE & ESS (low) Coal (low)            71678.0
VRE & ESS (mid) Coal (high)           71678.0
VRE & ESS (mid) Coal (low) - REF      71678.0
71678.0 71678.0
                                  capacity_mw
scenario                                     
VRE & ESS (high) Coal (high)       420748.543
VRE & ESS (high) Coal (low)        420748.543
VRE & ESS (low) Coal (high)        420748.543
VRE & ESS (low) Coal (low)         420748.543
VRE & ESS (mid) Coal (high)        420748.543
VRE & ESS (mid) Coal (low) - REF   420748.543
420748.54300018 420748.5430001801
5.869981626163957
       

In [ ]:
# What is the VRE share of the generation? 

In [98]:
df_ = pd.read_csv(path_to_tables + '/cost-grouped_dispatch.csv')
print(df_['scenario'].unique())

df_1_ = df_.loc[((df_['technology'] == 'Solar') 
                 | (df_['technology'] == 'Wind'))].reset_index(drop = True)

df_2_ = df_.loc[df_['power_mw'] > 0.].reset_index(drop = False)

df_1_ = df_1_.groupby(['period', 'scenario']).agg({'power_mw': 'sum'}).reset_index(drop = False)
df_2_ = df_2_.groupby(['period', 'scenario']).agg({'power_mw': 'sum'}).reset_index(drop = False)

df_1_.rename(columns = {'power_mw': 'vre_power_mw'}, inplace = True)
df_2_.rename(columns = {'power_mw': 'total_power_mw'}, inplace = True)

df_ = pd.merge(df_1_, df_2_, on=['period', 'scenario'], how = 'left')

df_['share'] = 100.*df_['vre_power_mw']/df_['total_power_mw']


df_ = df_.set_index('period')

#print(df_.loc[2020])
print(df_.loc[2020, 'share'].min(), df_.loc[2020, 'share'].max())

#print(df_.loc[2030])
print(df_.loc[2030, 'share'].min(), df_.loc[2030, 'share'].max())

#print(df_.loc[2040])
print(df_.loc[2040, 'share'].min(), df_.loc[2040, 'share'].max())

#print(df_.loc[2050])
print(df_.loc[2050, 'share'].min(), df_.loc[2050, 'share'].max())

['VRE & ESS (high) Coal (high)' 'VRE & ESS (high) Coal (low)'
 'VRE & ESS (low) Coal (high)' 'VRE & ESS (low) Coal (low)'
 'VRE & ESS (mid) Coal (high)' 'VRE & ESS (mid) Coal (low) - REF']
7.5364845360509225 7.5364845360509225
36.052086106603916 36.26933608642698
67.57253623937748 73.39815305739832
92.15047020277517 92.19804241023999


In [ ]:
# What is the solar PV share of VRE installed capacity? 

In [115]:
df_ = pd.read_csv(path_to_tables + '/cost-grouped_dispatch.csv')
print(df_['scenario'].unique())

df_ = df_.loc[(df_['technology'] == 'Solar') | (df_['technology'] == 'Wind')].reset_index(drop = True)

df_1_ = df_.groupby(['period', 'scenario']).agg({'power_mw': 'sum'}).reset_index(drop = False)
df_2_ = df_.groupby(['period', 'scenario', 'technology']).agg({'power_mw': 'sum'}).reset_index(drop = False)
df_1_.rename(columns = {'power_mw': 'total_power_mw'}, inplace = True)

df_ = pd.merge(df_1_, df_2_, on = ['period', 'scenario'], how = 'right')
df_['share'] = df_['power_mw']/df_['total_power_mw']

df_ = df_.set_index('period')
df_ = df_[df_['technology'] == 'Solar']

print(df_.loc[2020, 'share'].min(), df_.loc[2020, 'share'].max())
print(df_.loc[2030, 'share'].min(), df_.loc[2030, 'share'].max())
print(df_.loc[2040, 'share'].min(), df_.loc[2040, 'share'].max())
print(df_.loc[2050, 'share'].min(), df_.loc[2050, 'share'].max())

['VRE & ESS (high) Coal (high)' 'VRE & ESS (high) Coal (low)'
 'VRE & ESS (low) Coal (high)' 'VRE & ESS (low) Coal (low)'
 'VRE & ESS (mid) Coal (high)' 'VRE & ESS (mid) Coal (low) - REF']
0.43930468584982135 0.43930468584982135
0.40535126780265224 0.41713870655038426
0.5188618887603665 0.5506242955494999
0.5767367973534949 0.6444380439834795


In [ ]:
# What is the solar PV share of VRE generation? 

In [ ]:
df_ = pd.read_csv(path_to_tables + '/cost-grouped_capacity.csv')
print(df_['scenario'].unique())

df_ = df_.loc[(df_['technology'] == 'Solar') | (df_['technology'] == 'Wind')].reset_index(drop = True)

df_1_ = df_.groupby(['period', 'scenario']).agg({'capacity_mw': 'sum'}).reset_index(drop = False)
df_2_ = df_.groupby(['period', 'scenario', 'technology']).agg({'capacity_mw': 'sum'}).reset_index(drop = False)
df_1_.rename(columns = {'capacity_mw': 'total_capacity_mw'}, inplace = True)

df_ = pd.merge(df_1_, df_2_, on = ['period', 'scenario'], how = 'right')
df_['share'] = df_['capacity_mw']/df_['total_capacity_mw']

df_ = df_.set_index('period')
df_ = df_[df_['technology'] == 'Solar']

print(df_.loc[2020, 'share'].min(), df_.loc[2020, 'share'].max())
print(df_.loc[2030, 'share'].min(), df_.loc[2030, 'share'].max())
print(df_.loc[2040, 'share'].min(), df_.loc[2040, 'share'].max())
print(df_.loc[2050, 'share'].min(), df_.loc[2050, 'share'].max())

In [ ]:
# Which scenario builds offshore wind?

In [117]:
df_1_ = pd.read_csv(path_to_tables + '/cost-capacity.csv')
df_2_ = pd.read_csv(path_to_tables + '/pier-capacity.csv')
df_3_ = pd.read_csv(path_to_tables + '/iced-capacity.csv')
df_4_ = pd.read_csv(path_to_tables + '/demand-capacity.csv')
df_5_ = pd.read_csv(path_to_tables + '/alternative-capacity.csv')
df_6_ = pd.read_csv(path_to_tables + '/additional-capacity.csv')

df_1_['group'] = 'cost'
df_2_['group'] = 'pier'
df_3_['group'] = 'iced'
df_4_['group'] = 'demand'
df_5_['group'] = 'alternative'
df_6_['group'] = 'additional'

df_ = pd.concat([df_1_, df_2_, df_3_, df_4_, df_5_, df_6_], axis = 0).reset_index(drop = True)

df_ = df_.loc[df_['technology'] == 'Offshore'].reset_index(drop = True)

df_p_ = df_.groupby(['period', 'technology', 'group', 'scenario']).agg({'capacity_mw': 'sum'}).reset_index(drop = False)
print(df_p_.loc[df_p_['capacity_mw'] > 0])

     period technology   group                       scenario  capacity_mw
52     2030   Offshore  demand        Bottom-up Demand (high)  3458.389219
55     2030   Offshore  demand  Linearly-scaled Demand (high)  5757.750000
86     2040   Offshore  demand        Bottom-up Demand (high)  3458.389219
89     2040   Offshore  demand  Linearly-scaled Demand (high)  5757.750000
120    2050   Offshore  demand        Bottom-up Demand (high)  3458.389219
123    2050   Offshore  demand  Linearly-scaled Demand (high)  5757.750000


In [ ]:
# Do any demand scenarios encourage a VRE technology versus the other?

In [124]:
df_ = pd.read_csv(path_to_tables + '/demand-grouped_capacity.csv')
print(df_['scenario'].unique())

df_ = df_.loc[(df_['technology'] == 'Solar') | (df_['technology'] == 'Wind')].reset_index(drop = True)

df_1_ = df_.groupby(['period', 'scenario']).agg({'capacity_mw': 'sum'}).reset_index(drop = False)
df_2_ = df_.groupby(['period', 'scenario', 'technology']).agg({'capacity_mw': 'sum'}).reset_index(drop = False)
df_1_.rename(columns = {'capacity_mw': 'total_capacity_mw'}, inplace = True)

df_ = pd.merge(df_1_, df_2_, on = ['period', 'scenario'], how = 'right')
df_['share'] = df_['capacity_mw']/df_['total_capacity_mw']

df_ = df_.set_index('period')
df_ = df_[df_['technology'] == 'Solar']

print(df_.loc[2020])
print(df_.loc[2030])
print(df_.loc[2040])
print(df_.loc[2050])

['Linearly-scaled Demand (high)' 'Linearly-scaled Demand (low)'
 'Linearly-scaled Demand (mid)' 'Bottom-up Demand (high)'
 'Bottom-up Demand (low)' 'Bottom-up Demand (mid) - REF']
                             scenario  total_capacity_mw technology  \
period                                                                
2020          Bottom-up Demand (high)            71678.0      Solar   
2020           Bottom-up Demand (low)            71678.0      Solar   
2020     Bottom-up Demand (mid) - REF            71678.0      Solar   
2020    Linearly-scaled Demand (high)            71678.0      Solar   
2020     Linearly-scaled Demand (low)            71678.0      Solar   
2020     Linearly-scaled Demand (mid)            71678.0      Solar   

        capacity_mw     share  
period                         
2020        33963.0  0.473827  
2020        33963.0  0.473827  
2020        33963.0  0.473827  
2020        33963.0  0.473827  
2020        33963.0  0.473827  
2020        33963.0  0.4738

In [ ]:
# What is the battery power and energy capacity in the REF scenario?

In [128]:
df_ = pd.read_csv(path_to_tables + '/cost-grouped_capacity.csv')
print(df_['scenario'].unique())

df_ = df_.loc[df_['scenario'] == 'VRE & ESS (mid) Coal (low) - REF'].reset_index(drop = True)
df_ = df_.loc[df_['technology'] == 'Battery'].reset_index(drop = True)

df_ = df_.groupby(['period']).agg({'capacity_mw': 'sum', 'capacity_mwh': 'sum'}).reset_index(drop = False)
print(df_)


['VRE & ESS (high) Coal (high)' 'VRE & ESS (high) Coal (low)'
 'VRE & ESS (low) Coal (high)' 'VRE & ESS (low) Coal (low)'
 'VRE & ESS (mid) Coal (high)' 'VRE & ESS (mid) Coal (low) - REF']
   period    capacity_mw  capacity_mwh
0    2020       0.000000  0.000000e+00
1    2030  153981.823763  3.079636e+05
2    2040  400820.036526  8.016401e+05
3    2050  650111.745272  3.231283e+06


In [ ]:
# What is the battery energy capacity increase from 2030 to 2050 across VRE scenarios?

In [139]:
df_ = pd.read_csv(path_to_tables + '/cost-grouped_capacity.csv')
print(df_['scenario'].unique())

df_ = df_.loc[df_['technology'] == 'Battery'].reset_index(drop = True)

df_ = df_.groupby(['period', 'scenario']).agg({'capacity_mw': 'sum', 'capacity_mwh': 'sum'}).reset_index(drop = False)
print(df_)

x_1_ = df_.loc[df_['period'] == 2030, 'capacity_mwh'].to_numpy()
print(x_1_)

x_2_ = df_.loc[df_['period'] == 2050, 'capacity_mwh'].to_numpy()
print(x_2_)

print(x_2_/x_1_)
print((x_2_/x_1_).min(), (x_2_/x_1_).max())

['VRE & ESS (high) Coal (high)' 'VRE & ESS (high) Coal (low)'
 'VRE & ESS (low) Coal (high)' 'VRE & ESS (low) Coal (low)'
 'VRE & ESS (mid) Coal (high)' 'VRE & ESS (mid) Coal (low) - REF']
    period                          scenario    capacity_mw  capacity_mwh
0     2020      VRE & ESS (high) Coal (high)       0.000000  0.000000e+00
1     2020       VRE & ESS (high) Coal (low)       0.000000  0.000000e+00
2     2020       VRE & ESS (low) Coal (high)       0.000000  0.000000e+00
3     2020        VRE & ESS (low) Coal (low)       0.000000  0.000000e+00
4     2020       VRE & ESS (mid) Coal (high)       0.000000  0.000000e+00
5     2020  VRE & ESS (mid) Coal (low) - REF       0.000000  0.000000e+00
6     2030      VRE & ESS (high) Coal (high)  152864.259580  3.057285e+05
7     2030       VRE & ESS (high) Coal (low)  152864.259580  3.057285e+05
8     2030       VRE & ESS (low) Coal (high)  153332.320500  3.066646e+05
9     2030        VRE & ESS (low) Coal (low)  153332.320500  3.066646e+

In [ ]:
# What is the battery energy duration in 2020, 2030 and 2050?

In [142]:
df_ = pd.read_csv(path_to_tables + '/cost-grouped_capacity.csv')
print(df_['scenario'].unique())

df_ = df_.loc[df_['technology'] == 'Battery'].reset_index(drop = True)

df_ = df_.groupby(['period', 'scenario']).agg({'capacity_mw': 'sum', 'capacity_mwh': 'sum'}).reset_index(drop = False)
print(df_)

df_['duration'] = df_['capacity_mwh']/df_['capacity_mw']
print(df_['duration'])

['VRE & ESS (high) Coal (high)' 'VRE & ESS (high) Coal (low)'
 'VRE & ESS (low) Coal (high)' 'VRE & ESS (low) Coal (low)'
 'VRE & ESS (mid) Coal (high)' 'VRE & ESS (mid) Coal (low) - REF']
    period                          scenario    capacity_mw  capacity_mwh
0     2020      VRE & ESS (high) Coal (high)       0.000000  0.000000e+00
1     2020       VRE & ESS (high) Coal (low)       0.000000  0.000000e+00
2     2020       VRE & ESS (low) Coal (high)       0.000000  0.000000e+00
3     2020        VRE & ESS (low) Coal (low)       0.000000  0.000000e+00
4     2020       VRE & ESS (mid) Coal (high)       0.000000  0.000000e+00
5     2020  VRE & ESS (mid) Coal (low) - REF       0.000000  0.000000e+00
6     2030      VRE & ESS (high) Coal (high)  152864.259580  3.057285e+05
7     2030       VRE & ESS (high) Coal (low)  152864.259580  3.057285e+05
8     2030       VRE & ESS (low) Coal (high)  153332.320500  3.066646e+05
9     2030        VRE & ESS (low) Coal (low)  153332.320500  3.066646e+

In [ ]:
# What is the PSH and nuclear capacity in 2020? 

In [161]:
df_ = pd.read_csv(path_to_tables + '/alternative-grouped_capacity.csv')
print(df_['scenario'].unique())

df_p_ = df_.loc[(df_['scenario'] == 'Alternative (H2) - REF')].reset_index(drop = True)
df_p_ = df_p_.groupby(['period', 'technology']).agg({'capacity_mw': 'sum',
                                                     'capacity_mwh': 'sum'}).reset_index(drop = False)

print(df_p_.loc[(df_p_['technology'] == 'Nuclear')].reset_index(drop = True))
print(df_p_.loc[(df_p_['technology'] == 'Pumped Storage')].reset_index(drop = True))

df_p_ = df_.loc[(df_['scenario'] == 'Alternative (PSH)')].reset_index(drop = True)
df_p_ = df_p_.groupby(['period', 'technology']).agg({'capacity_mw': 'sum',
                                                     'capacity_mwh': 'sum'}).reset_index(drop = False)

print(df_p_.loc[(df_p_['technology'] == 'Pumped Storage')].reset_index(drop = True))

df_p_ = df_.loc[(df_['scenario'] == 'Alternative (all)')].reset_index(drop = True)
df_p_ = df_p_.groupby(['period', 'technology']).agg({'capacity_mw': 'sum',
                                                     'capacity_mwh': 'sum'}).reset_index(drop = False)

print(df_p_.loc[(df_p_['technology'] == 'Nuclear')].reset_index(drop = True))

['Alternative (all)' 'Alternative (H2 & PSH)' 'Alternative (H2) - REF'
 'Alternative (nuclear)' 'Alternative (PSH)' 'No Alternative']
   period technology  capacity_mw  capacity_mwh
0    2020    Nuclear       7880.0           0.0
1    2030    Nuclear      12680.0           0.0
2    2040    Nuclear      12680.0           0.0
3    2050    Nuclear      12680.0           0.0
   period      technology  capacity_mw  capacity_mwh
0    2020  Pumped Storage       3140.0       18840.0
1    2030  Pumped Storage       3140.0       18840.0
2    2040  Pumped Storage       3140.0       18840.0
3    2050  Pumped Storage       3140.0       18840.0
   period      technology    capacity_mw   capacity_mwh
0    2020  Pumped Storage    3140.000000   18840.000000
1    2030  Pumped Storage    3140.000000   18840.000000
2    2040  Pumped Storage   16246.253798  121217.115093
3    2050  Pumped Storage  117609.205320  876240.000000
   period technology   capacity_mw  capacity_mwh
0    2020    Nuclear   7880.0000

In [ ]:
# What is the total system cost in 2040 and 2050 in pier scenarios?

In [177]:
df_ = pd.read_csv(path_to_tables + '/pier-system_cost.csv')
print(df_['scenario'].unique())

df_['total_cost'] = df_['variable_cost'] + df_['fixed_cost'] + 5.9e9

z_ = df_.loc[df_['period'] == 2020, 'total_cost'].to_numpy()
x_ = df_.loc[df_['period'] == 2040, 'total_cost'].to_numpy()
y_ = df_.loc[df_['period'] == 2050, 'total_cost'].to_numpy()
print(x_.min(), x_.max())
print(y_.min(), y_.max())
print((y_/z_).min(), (y_/z_).max())

['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_80CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_100CAP_500GW_PIERmid']
['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_80CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_100CAP_500GW_PIERmid']
189564895989.8323 191643189739.0705
245568490264.20877 299540390903.1568
2.8091493360076956 3.4265539903257247


In [ ]:
# How much low VRE and ST reduce emissions with enforcing a carbon target?

In [181]:
df_ = pd.read_csv(path_to_tables + '/additional-emissions.csv')
print(df_['scenario'].unique())

df_ = df_.loc[(df_['scenario'] == 'VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid')].reset_index(drop = True)
print(df_)
A = df_.loc[df_['period'] == 2020, 'carbon_emissions_tons'].to_numpy()
B = df_.loc[df_['period'] == 2040, 'carbon_emissions_tons'].to_numpy()
C = df_.loc[df_['period'] == 2050, 'carbon_emissions_tons'].to_numpy()
print(A, B, C)
print(1. - B/A, 1. - C/A)

['VREhigh_SThigh_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'
 'VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAPLin_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid']
   period                                           scenario load_zone  \
0    2020  VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_500G...       all   
1    2030  VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_500G...       all   
2    2040  VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_500G...       all   
3    2050  VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_500G...       all   

   carbon_emissions_tons       load_mw  
0           1.119110e+09  1.535536e+09  
1           1.256424e+09  2.582122e+09  
2           7.593366e+08  4.188931e+09  
3           6.021931e+08  5.622802e+

In [ ]:
# Do the shares of technology costs in absolute and relative terms are the same?

In [193]:
df_1_ = pd.read_csv(path_to_tables + '/cost-technology_costs.csv')
df_2_ = pd.read_csv(path_to_tables + '/cost-emissions.csv').drop(columns = ['load_zone', 'carbon_emissions_tons'])
print(df_1_['scenario'].unique())

df_1_ = df_1_.groupby(['period', 'scenario']).agg({'variable_cost': 'sum',
                                                   'fixed_cost': 'sum'}).reset_index(drop = False)

df_ = pd.merge(df_1_, df_2_, on = ['period', 'scenario'], how = 'right')

df_['share_variable_cost_1'] = df_['variable_cost']/(df_['variable_cost'] + df_['fixed_cost'])
df_['share_variable_cost_2'] = (df_['variable_cost']/df_['load_mw'])/((df_['variable_cost'] + df_['fixed_cost'])/df_['load_mw'])
print(df_)

['VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VRElow_STlow_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREhigh_SThigh_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREhigh_SThigh_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid']
    period                                           scenario  variable_cost  \
0     2020  VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...   3.944529e+10   
1     2030  VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...   4.657126e+10   
2     2040  VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...   2.967787e+10   
3     2050  VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...   8.789741e+09   
4     2020  VRElow_STlow_CONVhigh_H2_RES_8PRM_CC_50RPS_90C...   3.944529e+10   
5     2030  VRElow_STlow_CONVhigh_H2_RES_8PRM_CC_50RPS_90C...   4.657126e+10   
6     2040  VRElow_STlow_CONVhigh_H2_RES_8

In [ ]:
# Does the share of variable cost in the cost of coal decreases?

In [203]:
df_ = pd.read_csv(path_to_tables + '/cost-technology_costs.csv')
print(df_['scenario'].unique())
print(df_['technology'].unique())

df_ = df_.loc[df_['technology'] == 'Coal'].reset_index(drop = True)
print(df_)
df_ = df_.groupby(['period', 'scenario']).agg({'variable_cost': 'sum',
                                               'fixed_cost': 'sum'}).reset_index(drop = False)

df_['total_cost'] = df_['variable_cost'] + df_['fixed_cost']
df_['share_variable_cost'] = df_['variable_cost'] / df_['total_cost']

x_ = df_.loc[df_['period'] == 2020, 'share_variable_cost'].to_numpy()
y_ = df_.loc[df_['period'] == 2030, 'share_variable_cost'].to_numpy()
z_ = df_.loc[df_['period'] == 2040, 'share_variable_cost'].to_numpy()
w_ = df_.loc[df_['period'] == 2050, 'share_variable_cost'].to_numpy()

print(x_.min(), x_.max())
print(y_.min(), y_.max())
print(z_.min(), z_.max())
print(w_.min(), w_.max())

['VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VRElow_STlow_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREhigh_SThigh_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREhigh_SThigh_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid']
['Battery' 'Coal' 'Diesel' 'Gas' 'Hydro' 'Hydrogen' 'Nuclear' 'Other'
 'Pumped Storage' 'Solar' 'Transmission Losses' 'Wind']
    period technology                                           scenario  \
0     2020       Coal  VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...   
1     2030       Coal  VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...   
2     2040       Coal  VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...   
3     2050       Coal  VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...   
4     2020       Coal  VRElow_STlow_CONVhigh_H2_RES_8PRM_CC_50RPS_90C...   
5     2030       Coa

This is just the share of solar capacity within the VRE capacity across all scenarios and investment periods. I am guessing it is similar. Please add the share range for scenarios for each investment period here in the comment. We can then decide whether we want to combine all the numbers and give just one range. 

In [169]:
df_1_ = pd.read_csv(path_to_tables + '/cost-grouped_capacity.csv')
df_2_ = pd.read_csv(path_to_tables + '/pier-grouped_capacity.csv')
df_3_ = pd.read_csv(path_to_tables + '/iced-grouped_capacity.csv')
df_4_ = pd.read_csv(path_to_tables + '/demand-grouped_capacity.csv')
df_5_ = pd.read_csv(path_to_tables + '/alternative-grouped_capacity.csv')
df_6_ = pd.read_csv(path_to_tables + '/additional-grouped_capacity.csv')

df_1_['group'] = 'cost'
df_2_['group'] = 'pier'
df_3_['group'] = 'iced'
df_4_['group'] = 'demand'
df_5_['group'] = 'alternative'
df_6_['group'] = 'additional'

df_ = pd.concat([df_1_, df_2_, df_3_, df_4_, df_5_, df_6_], axis = 0).reset_index(drop = True)
print(df_.shape)

df_ = df_.drop_duplicates(keep = 'first')
print(df_.shape)

df_p_ = df_.loc[(df_['technology'] == 'Solar') | (df_['technology'] == 'Wind')].reset_index(drop = True)

df_p_ = df_p_.groupby(['period', 'technology', 'scenario', 'group']).agg({'capacity_mw': 'sum'}).reset_index(drop = False)
print(df_p_.shape)

df_ = []
for period in df_p_['period'].unique():
    df_pp_ = df_p_.loc[df_p_['period'] == period]
    
    for scenario in df_pp_['scenario'].unique():
        df_ppp_ = df_pp_.loc[df_pp_['scenario'] == scenario]
        
        for group in df_ppp_['group'].unique():

            df_pppp_ = df_ppp_.loc[df_ppp_['group'] == group]

            group  = df_pppp_['group'].to_numpy()[0]
            total  = df_pppp_['capacity_mw'].sum()
            solar  = df_pppp_.loc[(df_pppp_['technology'] == 'Solar'), 'capacity_mw'].to_numpy()[0]/total
            wind   = df_pppp_.loc[(df_pppp_['technology'] == 'Wind'), 'capacity_mw'].to_numpy()[0]/total
            df_.append([period, scenario, group, solar, wind, total])

df_ = pd.DataFrame(df_, columns = ['period', 'scenario', 'group', 'solar', 'wind', 'capacity_mw'])
df_['solar_min'] = df_['scenario']
df_['solar_max'] = df_['solar']

df_p_ = df_.groupby(['period']).agg({'solar_min': 'min', 
                                     'solar_max': 'max'}).reset_index(drop = False)
print(df_p_)

(33336, 10)
(33336, 10)
(264, 5)
   period  solar_min  solar_max
0    2020   0.473827   0.473827
1    2030   0.548490   0.697162
2    2040   0.553777   0.723865
3    2050   0.596930   0.801474


Add result of offshore wind. e.g. Onshore (land-based) wind remains the dominant deployment among wind technologies, 
Offshore wind, based on relative costs and available resources, has a relatively small share of wind deployment---xx\% and xx\% by 2040 and 2050, respectively.  

In [184]:
df_1_ = pd.read_csv(path_to_tables + '/cost-capacity.csv')
df_2_ = pd.read_csv(path_to_tables + '/pier-capacity.csv')
df_3_ = pd.read_csv(path_to_tables + '/iced-capacity.csv')
df_4_ = pd.read_csv(path_to_tables + '/demand-capacity.csv')
df_5_ = pd.read_csv(path_to_tables + '/alternative-capacity.csv')
df_6_ = pd.read_csv(path_to_tables + '/additional-capacity.csv')

df_1_['group'] = 'cost'
df_2_['group'] = 'pier'
df_3_['group'] = 'iced'
df_4_['group'] = 'demand'
df_5_['group'] = 'alternative'
df_6_['group'] = 'additional'

df_ = pd.concat([df_1_, df_2_, df_3_, df_4_, df_5_, df_6_], axis = 0).reset_index(drop = True)

df_ = df_.loc[(df_['technology'] == 'Wind') | (df_['technology'] == 'Offshore')].reset_index(drop = True)

df_p_ = df_.groupby(['period', 'technology', 'group', 'scenario']).agg({'capacity_mw': 'sum'}).reset_index(drop = False)
print(df_)

df_ = []
for period in df_p_['period'].unique():
    df_pp_ = df_p_.loc[df_p_['period'] == period]
    
    for scenario in df_pp_['scenario'].unique():
        df_ppp_ = df_pp_.loc[df_pp_['scenario'] == scenario]
        
        for group in df_ppp_['group'].unique():

            df_pppp_ = df_ppp_.loc[df_ppp_['group'] == group]

            group  = df_pppp_['group'].to_numpy()[0]
            total  = df_pppp_['capacity_mw'].sum()
            offshore  = df_pppp_.loc[(df_pppp_['technology'] == 'Offshore'), 'capacity_mw'].to_numpy()[0]/total
            wind   = df_pppp_.loc[(df_pppp_['technology'] == 'Wind'), 'capacity_mw'].to_numpy()[0]/total
            df_.append([period, scenario, group, offshore, wind, total])

df_ = pd.DataFrame(df_, columns = ['period', 'scenario', 'group', 'offshore', 'wind', 'capacity_mw'])
df_['offshore_min'] = df_['offshore']
df_['offshore_max'] = df_['offshore']

df_p_ = df_.groupby(['period']).agg({'offshore_min': 'min', 
                                     'offshore_max': 'max'}).reset_index(drop = False)
print(df_p_)

      period technology           load_zone    status  capacity_mw  \
0       2020   Offshore      Andhra_Pradesh       new      0.00000   
1       2020       Wind      Andhra_Pradesh  existing   4090.00000   
2       2020       Wind      Andhra_Pradesh       new      0.00000   
3       2030   Offshore      Andhra_Pradesh       new      0.00000   
4       2030       Wind      Andhra_Pradesh  existing   4090.00000   
...      ...        ...                 ...       ...          ...   
4615    2050       Wind  Dadra_Nagar_Haveli       new      0.00000   
4616    2020       Wind  Dadra_Nagar_Haveli       new      0.00000   
4617    2030       Wind  Dadra_Nagar_Haveli       new      0.00000   
4618    2040       Wind  Dadra_Nagar_Haveli       new      0.00000   
4619    2050       Wind  Dadra_Nagar_Haveli       new    685.93699   

      capacity_mwh                                           scenario  \
0              0.0                         VRE & ESS (low) Coal (low)   
1            

In [191]:
df_ = pd.read_csv(path_to_tables + '/cost-capacity.csv')
print(df_['scenario'].unique())

df_ = df_.loc[df_['period'] == 2050]
df_ = df_.loc[df_['scenario'] == 'VRE & ESS (mid) Coal (low) - REF']


df_ = df_.groupby(['period', 'technology']).agg({'capacity_mw': 'sum'}).reset_index(drop = False)

print(df_)

['VRE & ESS (low) Coal (low)' 'VRE & ESS (low) Coal (high)'
 'VRE & ESS (mid) Coal (low) - REF' 'VRE & ESS (mid) Coal (high)'
 'VRE & ESS (high) Coal (low)' 'VRE & ESS (high) Coal (high)']
    period              technology   capacity_mw
0     2050                 Battery  6.501117e+05
1     2050                 Biomass  1.020334e+04
2     2050                    CCGT  2.450536e+04
3     2050                      CT  0.000000e+00
4     2050                  Diesel  1.213960e+03
5     2050            Hydro_Pumped  3.140000e+03
6     2050               Hydro_ROR  3.542817e+04
7     2050           Hydro_Storage  2.084245e+04
8     2050                Hydrogen  3.749357e+03
9     2050                 Nuclear  1.268000e+04
10    2050                Offshore  0.000000e+00
11    2050            SolarPV_roof  3.017114e+05
12    2050          SolarPV_single  1.663799e+06
13    2050            SolarPV_tilt  1.734179e+05
14    2050  Subcritical_Coal_Large  4.023000e+04
15    2050  Subcritical_Coa

### Battery storage

In [127]:
df_ = pd.read_csv(path_to_tables + '/cost-grouped_capacity.csv')

df_ = df_.loc[(df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid')].reset_index(drop = True)
df_ = df_.loc[(df_['technology'] == 'Battery')].reset_index(drop = True)

df_ = df_.groupby(['period']).agg({'capacity_mw': 'sum',
                                   'capacity_mwh': 'sum'})#.reset_index(drop = False)
print(df_/1000.)
print(df_.loc[2050, 'capacity_mw']/df_.loc[2030, 'capacity_mw'], df_.loc[2050, 'capacity_mwh']/df_.loc[2030, 'capacity_mwh'])

        capacity_mw  capacity_mwh
period                           
2020       0.000000      0.000000
2030     153.981824    307.963648
2040     400.820037    801.640073
2050     650.111745   3231.283435
4.222003151972859 10.492418379196128


In [73]:
df_ = pd.read_csv(path_to_tables + '/cost-grouped_dispatch.csv')

print(df_['scenario'].unique())

print(df_['technology'].unique())

df_ = df_.groupby(['period', 'scenario', 'technology']).agg({'power_mw': 'sum'}).reset_index(drop = False)
df_ = df_.loc[df_['power_mw'] > 0.].reset_index(drop = False)
df_p_ = df_.groupby(['period', 'scenario']).agg({'power_mw': 'sum'}).reset_index(drop = False)
df_p_.rename(columns = {'power_mw': 'total_generation_mw'}, inplace = True)

df_ = df_.loc[(df_['technology'] == 'Solar') | (df_['technology'] == 'Wind')].reset_index(drop = False)

df_pp_ = df_.groupby(['period', 'scenario']).agg({'power_mw': 'sum'}).reset_index(drop = False)
df_pp_.rename(columns = {'power_mw': 'total_vre_generation_mw'}, inplace = True)

df_ = pd.merge(df_p_, df_pp_, on=['period', 'scenario'], how='inner')
df_['vre_frac'] = 100.*df_['total_vre_generation_mw']/df_['total_generation_mw']
print(df_)


['VRE & ESS (high) Coal (high)' 'VRE & ESS (high) Coal (low)'
 'VRE & ESS (low) Coal (high)' 'VRE & ESS (low) Coal (low)'
 'VRE & ESS (mid) Coal (high)' 'VRE & ESS (mid) Coal (low) - REF']
['Battery' 'Coal' 'Curtailment' 'Gas' 'Hydro' 'Other' 'Shedding' 'Solar'
 'Transmission Losses' 'Wind' 'Hydrogen' 'Diesel' 'Nuclear'
 'Pumped Storage']
    period                          scenario  total_generation_mw  \
0     2020      VRE & ESS (high) Coal (high)         1.554360e+09   
1     2020       VRE & ESS (high) Coal (low)         1.554360e+09   
2     2020       VRE & ESS (low) Coal (high)         1.554360e+09   
3     2020        VRE & ESS (low) Coal (low)         1.554360e+09   
4     2020       VRE & ESS (mid) Coal (high)         1.554360e+09   
5     2020  VRE & ESS (mid) Coal (low) - REF         1.554360e+09   
6     2030      VRE & ESS (high) Coal (high)         2.628363e+09   
7     2030       VRE & ESS (high) Coal (low)         2.628363e+09   
8     2030       VRE & ESS (low) Coal 

### Long-duration hydrogen storage

In [198]:
df_ = pd.read_csv(path_to_tables + '/cost-grouped_capacity.csv')
print(df_['scenario'].unique())

df_ = df_.loc[df_['technology'] == 'Hydrogen']
#df_ = df_.loc[df_['scenario'] == 'VRE & ESS (mid) Coal (low) - REF']


df_ = df_.groupby(['period', 'scenario']).agg({'capacity_mw': 'sum',
                                               'capacity_mwh': 'sum'}).reset_index(drop = False)

df_['capacity_mw']  /= 1000.
df_['capacity_mwh'] /= 1000.

print(df_)

['VRE & ESS (high) Coal (high)' 'VRE & ESS (high) Coal (low)'
 'VRE & ESS (low) Coal (high)' 'VRE & ESS (low) Coal (low)'
 'VRE & ESS (mid) Coal (high)' 'VRE & ESS (mid) Coal (low) - REF']
    period                          scenario  capacity_mw  capacity_mwh
0     2020      VRE & ESS (high) Coal (high)     0.000000  0.000000e+00
1     2020       VRE & ESS (high) Coal (low)     0.000000  0.000000e+00
2     2020       VRE & ESS (low) Coal (high)     0.000000  0.000000e+00
3     2020        VRE & ESS (low) Coal (low)     0.000000  0.000000e+00
4     2020       VRE & ESS (mid) Coal (high)     0.000000  0.000000e+00
5     2020  VRE & ESS (mid) Coal (low) - REF     0.000000  0.000000e+00
6     2030      VRE & ESS (high) Coal (high)     0.000000  0.000000e+00
7     2030       VRE & ESS (high) Coal (low)     0.000000  0.000000e+00
8     2030       VRE & ESS (low) Coal (high)     0.000000  0.000000e+00
9     2030        VRE & ESS (low) Coal (low)     0.000000  0.000000e+00
10    2030       VR

In [218]:
df_ = pd.read_csv(path_to_tables + '/cost-system_cost.csv')

idx_ = ((df_['period'] == 2050) 
         & (df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'))
B = df_.loc[idx_, 'variable_cost'] + df_.loc[idx_, 'fixed_cost']
print(df_.loc[idx_])  


    period                                           scenario load_zone  \
11    2050  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...       all   

    variable_cost    fixed_cost  
11   9.768498e+09  2.600698e+11  


In [255]:
df_ = pd.read_csv(path_to_tables + '/alternative-system_cost.csv')
print(df_['scenario'].unique())

idx_ = ((df_['period'] == 2050) 
         & (df_['scenario'] == 'VREmid_STmid_CONVmid_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'))
print(df_.loc[idx_])    
A = df_.loc[idx_, 'variable_cost'] + df_.loc[idx_, 'fixed_cost']        
print(A)

idx_ = ((df_['period'] == 2050) 
         & (df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'))
B = df_.loc[idx_, 'variable_cost'] + df_.loc[idx_, 'fixed_cost']
print(df_.loc[idx_])    

print(B)

C = 100.*B.to_numpy()[0]/A.to_numpy()[0]
print(C)

['VREmid_STmid_CONVmid_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_PSH_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2PSH_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_N_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2PSHN_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid']
   period                                           scenario load_zone  \
3    2050  VREmid_STmid_CONVmid_RES_8PRM_CC_50RPS_90CAP_5...       all   

   variable_cost    fixed_cost  
3   9.033427e+09  2.584824e+11  
3    2.675159e+11
dtype: float64
   period                                           scenario load_zone  \
7    2050  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...       all   

   variable_cost    fixed_cost  
7   9.021275e+09  2.584760e+11  
7    2.674973e+11
dtype: float64
99.99306110428775


Nuclear and pumped-hydro storage (PSH) are two other commercially available technologies that could play a role in India's future low carbon electricity system. However, expanding nuclear beyond xxx and scaling up PSH above xxx have only a small effect on total costs in 2050 (xx-xx\%) 

In [86]:
df_ = pd.read_csv(path_to_tables + '/alternative-grouped_capacity.csv')
print(df_['scenario'].unique())

df_ = df_.groupby(['period', 'technology', 'scenario']).agg({'capacity_mw': 'sum',
                                                             'capacity_mwh': 'sum'}).reset_index(drop = False)

idx_ = (df_['period'] == 2050) & (df_['scenario'] == 'Alternative (PSH)') & (df_['technology'] == 'Pumped Storage')
print(df_.loc[idx_])    

idx_ = (df_['period'] == 2050) & (df_['scenario'] == 'Alternative (nuclear)') & (df_['technology'] == 'Nuclear')
print(df_.loc[idx_])    

['Alternative (all)' 'Alternative (H2 & PSH)' 'Alternative (H2) - REF'
 'Alternative (nuclear)' 'Alternative (PSH)' 'No Alternative']
     period      technology           scenario   capacity_mw  capacity_mwh
236    2050  Pumped Storage  Alternative (PSH)  117609.20532      876240.0
     period technology               scenario   capacity_mw  capacity_mwh
226    2050    Nuclear  Alternative (nuclear)  71805.289403           0.0


In [89]:
df_ = pd.read_csv(path_to_tables + '/alternative-system_cost.csv')
print(df_['scenario'].unique())

idx_ = (df_['period'] == 2050) & (df_['scenario'] == 'VREmid_STmid_CONVmid_N_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid')
A = df_.loc[idx_, 'variable_cost'] + df_.loc[idx_, 'fixed_cost']        
print(df_.loc[idx_])    

idx_ = (df_['period'] == 2050) & (df_['scenario'] == 'VREmid_STmid_CONVmid_PSH_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid')
B = df_.loc[idx_, 'variable_cost'] + df_.loc[idx_, 'fixed_cost']
print(df_.loc[idx_])    

idx_ = (df_['period'] == 2050) & (df_['scenario'] == 'VREmid_STmid_CONVmid_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid')
C = df_.loc[idx_, 'variable_cost'] + df_.loc[idx_, 'fixed_cost']
print(df_.loc[idx_])    

print(A.to_numpy()[0], B.to_numpy()[0], C.to_numpy()[0])
print(100.*(1. - A.to_numpy()[0]/C.to_numpy()[0]), 100.*(1. - B.to_numpy()[0]/C.to_numpy()[0]))

['VREmid_STmid_CONVmid_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_PSH_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2PSH_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_N_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2PSHN_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid']
    period                                           scenario load_zone  \
19    2050  VREmid_STmid_CONVmid_N_RES_8PRM_CC_50RPS_90CAP...       all   

    variable_cost    fixed_cost  
19   1.542885e+10  2.513872e+11  
    period                                           scenario load_zone  \
11    2050  VREmid_STmid_CONVmid_PSH_RES_8PRM_CC_50RPS_90C...       all   

    variable_cost    fixed_cost  
11   9.039158e+09  2.555628e+11  
   period                                           scenario load_zone  \
3    2050  VREmid_STmid_CONVmid_RES_8PRM_CC_50RPS_90CAP_5...       all   

   variable_cost  

In [251]:
df_1_ = pd.read_csv(path_to_tables + '/cost-demand.csv')

idx_ = ((df_1_['type'] == 'Demand')
        & (df_1_['load_zone'] == 'Delhi')
        & (df_1_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'))

df_1_ = df_1_.loc[idx_]
print(df_1_['power_mw'].to_numpy())

df_2_ = pd.read_csv(path_to_tables + '/cost-grouped_dispatch.csv')

idx_ = ((df_2_['load_zone'] == 'Delhi')
        & (df_2_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'))


df_2_ = df_2_.loc[idx_].groupby(['period', 
                               'load_zone', 
                               'scenario']).agg({'power_mw': 'sum'}).reset_index(drop = False)

print(df_2_['power_mw'].to_numpy())

print(100.*df_2_['power_mw'].to_numpy()/df_1_['power_mw'].to_numpy())

[3.73364770e+07 6.01866318e+07 9.83366932e+07 1.37732540e+08]
[ 7011090.78796495  7725633.2114456   9526931.86216084 11073375.84249466]
[18.77812625 12.83612818  9.68807426  8.03976742]


In [257]:
df_ = pd.read_csv(path_to_tables + '/additional-system_cost.csv')
print(df_['scenario'].unique())

idx_ = ((df_['period'] == 2050) 
         & (df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'))
print(df_.loc[idx_])    
A = df_.loc[idx_, 'variable_cost'] + df_.loc[idx_, 'fixed_cost']        
print(A)

idx_ = ((df_['period'] == 2050) 
         & (df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_90CAP_500GW_PIERmid'))
B = df_.loc[idx_, 'variable_cost'] + df_.loc[idx_, 'fixed_cost']
print(df_.loc[idx_])    

print(B)

C = 100.*B.to_numpy()[0]/A.to_numpy()[0]
print(C)

['VREhigh_SThigh_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAPLin_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_50RPS_90CAP_500GW_PIERmid']
    period                                           scenario load_zone  \
11    2050  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...       all   

    variable_cost    fixed_cost  
11   9.021275e+09  2.584760e+11  
11    2.674973e+11
dtype: float64
    period                                           scenario load_zone  \
15    2050  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_90CAP_500G...       all   

    variable_cost    fixed_cost  
15   9.020853e+09  2.585367e+11  
15    2.675576e+11
dtype: float64
100.02253049479566


## Changing costs with increased decarbonization

In [272]:
df_ = pd.read_csv(path_to_tables + '/cost-system_cost.csv')
print(df_['scenario'].unique())

period = 2050

idx_high_ = ((df_['scenario'] == 'VREhigh_SThigh_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid')
             & (df_['period'] == period))

idx_mid_ = ((df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid')
            & (df_['period'] == period))

idx_low_ = ((df_['scenario'] == 'VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid')
            & (df_['period'] == period))

cost_high = (df_.loc[idx_high_, 'variable_cost'] + df_.loc[idx_high_, 'fixed_cost']).to_numpy()
cost_mid  = (df_.loc[idx_mid_, 'variable_cost']  + df_.loc[idx_mid_, 'fixed_cost']).to_numpy()
cost_low  = (df_.loc[idx_low_, 'variable_cost']  + df_.loc[idx_low_, 'fixed_cost']).to_numpy()
print(cost_high, cost_mid, cost_low)

print(100.*cost_high/cost_mid - 100, 100 -100.*cost_low/cost_mid)

['VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VRElow_STlow_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREhigh_SThigh_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREhigh_SThigh_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid']
[3.16581852e+11] [2.67497303e+11] [2.08762307e+11]
[18.34954907] [21.95722913]


[ADD if true: Lower VRE and storage cost projections also allow meeting high carbon emission reductions without any carbon target (xx\% by 2050), and thus imposes only limited additional costs to meet a 90\% carbon mitigation target by 2050.]

In [98]:
df_ = pd.read_csv(path_to_tables + '/additional-emissions.csv')
print(df_['scenario'].unique())

period = 2050

idx_  = (df_['scenario'] == 'VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid') 
df_p_ = df_.loc[idx_].reset_index(drop = True)
print(df_p_)
print(1. - df_p_['carbon_emissions_tons'].to_numpy()[-1]/df_['carbon_emissions_tons'].to_numpy()[0])

idx_ = (df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAPLin_500GW_PIERmid')
df_p_ = df_.loc[idx_].reset_index(drop = True)
print(df_p_)
print(1. - df_p_['carbon_emissions_tons'].to_numpy()[-1]/df_['carbon_emissions_tons'].to_numpy()[0])

# # idx_low_ = ((df_['scenario'] == 'VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid')
# #             & (df_['period'] == period))
# # print(df_.loc[(df_['period'] == period)])
# # print(df_.loc[idx_high_, 'carbon_emissions_tons']/1e9)
# # print(df_.loc[idx_mid_, 'carbon_emissions_tons']/1e9)

# # cost_high = (df_.loc[idx_high_, 'variable_cost'] + df_.loc[idx_high_, 'fixed_cost']).to_numpy()
# # cost_mid  = (df_.loc[idx_mid_, 'variable_cost']  + df_.loc[idx_mid_, 'fixed_cost']).to_numpy()
# # cost_low  = (df_.loc[idx_low_, 'variable_cost']  + df_.loc[idx_low_, 'fixed_cost']).to_numpy()
# # print(cost_high, cost_mid, cost_low)


['VREhigh_SThigh_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'
 'VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAPLin_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_50RPS_90CAP_500GW_PIERmid']
   period                                           scenario load_zone  \
0    2020  VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_500G...       all   
1    2030  VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_500G...       all   
2    2040  VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_500G...       all   
3    2050  VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_500G...       all   

   carbon_emissions_tons       load_mw  
0           1.119110e+09  1.535536e+09  
1           1.256424e+09  2.582122e+09  
2           7.593366e+08  4.188931e+09  
3           6.021931e+08  5.622802e+09  
0.4619001966192261
   period                          

Out of total system costs, the share of variable costs decreases from xx\% in 2020 to xx-xx\% in 2030, xx-xx\% in 2040, and xx-xx\% in 2050.

In [122]:
df_ = pd.read_csv(path_to_tables + '/cost-technology_costs.csv')
print(df_['scenario'].unique())

df_ = df_.groupby(['period', 'scenario']).agg({'variable_cost': 'sum',
                                               'fixed_cost': 'sum'}).reset_index(drop = False)

df_['total_costs']         = df_['fixed_cost'] + df_['variable_cost']
df_['variable_cost_share'] = 100.*df_['variable_cost']/df_['total_costs']
print(df_)

df_p_ = df_.groupby(['period']).agg({'variable_cost_share': 'min'}).reset_index(drop = False)
print(df_p_.round(1))
df_p_ = df_.groupby(['period']).agg({'variable_cost_share': 'max'}).reset_index(drop = False)
print(df_p_.round(1))

['VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VRElow_STlow_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREhigh_SThigh_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREhigh_SThigh_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid']
    period                                           scenario  variable_cost  \
0     2020  VREhigh_SThigh_CONVhigh_H2_RES_8PRM_CC_50RPS_9...   3.944529e+10   
1     2020  VREhigh_SThigh_CONVmid_H2_RES_8PRM_CC_50RPS_90...   3.944529e+10   
2     2020  VRElow_STlow_CONVhigh_H2_RES_8PRM_CC_50RPS_90C...   3.944529e+10   
3     2020  VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...   3.944529e+10   
4     2020  VREmid_STmid_CONVhigh_H2_RES_8PRM_CC_50RPS_90C...   3.944529e+10   
5     2020  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...   3.944529e+10   
6     2030  VREhigh_SThigh_CONVhigh_H2_RES

## Transmission and electricity trade

In [300]:
df_ = pd.read_csv(path_to_tables + '/cost-tx_capacity.csv')
df_ = df_.groupby(['period', 'scenario']).agg({'spec_mw': 'sum',
                                               'new_mw': 'sum'}).reset_index(drop = False)

idx_ = df_['scenario'] == 'VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
print(df_.loc[idx_, 'new_mw']/df_.loc[idx_, 'spec_mw'])

idx_ = df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
print(df_.loc[idx_]['new_mw']/df_.loc[idx_]['spec_mw'])

idx_ = df_['scenario'] == 'VREhigh_SThigh_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
print(df_.loc[idx_, 'new_mw']/df_.loc[idx_, 'spec_mw'])

3     0.000000
9     0.002501
15    0.067061
21    0.264348
dtype: float64
5     0.000000
11    0.002501
17    0.065370
23    0.323150
dtype: float64
1     0.000000
7     0.002501
13    0.068304
19    0.370345
dtype: float64


In [308]:
df_ = pd.read_csv(path_to_tables + '/cost-total_tx_transfers.csv')

df_['power_mw'] = df_['power_mw'].abs()/1e-6
df_ = df_.groupby(['period', 
                   'scenario']).agg({'power_mw': 'sum'}).reset_index(drop = False)

idx_ = df_['scenario'] == 'VRE & ESS (low) Coal (low)'
print(df_.loc[idx_, 'power_mw'].to_numpy()/df_.loc[idx_, 'power_mw'].to_numpy()[0])

idx_ = df_['scenario'] == 'VRE & ESS (mid) Coal (low) - REF'
print(df_.loc[idx_, 'power_mw'].to_numpy()/df_.loc[idx_, 'power_mw'].to_numpy()[0])

idx_ = df_['scenario'] == 'VRE & ESS (high) Coal (low)'
print(df_.loc[idx_, 'power_mw'].to_numpy()/df_.loc[idx_, 'power_mw'].to_numpy()[0])

[1.         1.83362036 3.00762584 4.36410399]
[1.         1.84710703 2.80961528 4.49932496]
[1.         1.84306628 2.7783968  4.82076957]


# Discussion 

## Official near-term 2030 clean energy targets are lower bounds

In the near-term (2030), over xx GW of non-fossil capacity is deployed cost-optimally. Thus, the 500~GW non-fossil capacity target is a lower bound and aiming for a higher target would decrease costs and emissions in the near term.

In [108]:
df_ = pd.read_csv(path_to_tables + '/pier-grouped_capacity.csv')
print(df_['scenario'].unique())

df_p_ = df_.loc[df_['scenario'] == 'No Clean Target or Carbon Target']
df_p_ = df_p_.loc[df_['period'] == 2030]

df_p_ = df_p_.groupby(['technology']).agg({'capacity_mw': 'sum',
                                           'capacity_mwh': 'sum'})
print(df_p_.loc['Solar', 'capacity_mw'])
print(df_p_.loc['Wind', 'capacity_mw'])
print(df_p_.loc['Solar', 'capacity_mw'] + df_p_.loc['Wind', 'capacity_mw'])

df_pp_ = df_.loc[df_['scenario'] == '90% Carbon Target - REF']
df_pp_ = df_pp_.loc[df_['period'] == 2030]

df_pp_ = df_pp_.groupby(['technology']).agg({'capacity_mw': 'sum',
                                             'capacity_mwh': 'sum'})
print(df_pp_.loc['Solar', 'capacity_mw'])
print(df_pp_.loc['Wind', 'capacity_mw'])

['100% Carbon Target' 'No Carbon Target' '80% Carbon Target'
 '90% Carbon Target - REF' 'No Clean Target or Carbon Target']
494128.2425955034
214642.96830624927
708771.2109017527
246365.8541439764
174382.68885620366


In [219]:
df_ = pd.read_csv(path_to_tables + '/cost-grouped_capacity.csv')
print(df_['technology'].unique())

df_1_ = df_.loc[df_['technology'] == 'Battery'].copy()
df_1_ = df_1_.loc[df_1_['scenario'] == 'VRE & ESS (mid) Coal (low) - REF']


df_1_ = df_1_.groupby(['period', 'scenario']).agg({'capacity_mw': 'sum',
                                                   'capacity_mwh': 'sum'}).reset_index(drop = False)

df_1_['capacity_mw']  /= 1000.
df_1_['capacity_mwh'] /= 1000.
print(df_1_)

['Battery' 'Coal' 'Gas' 'Hydro' 'Other' 'Solar' 'Wind' 'Hydrogen' 'Diesel'
 'Nuclear' 'Pumped Storage']
   period                          scenario  capacity_mw  capacity_mwh
0    2020  VRE & ESS (mid) Coal (low) - REF     0.000000      0.000000
1    2030  VRE & ESS (mid) Coal (low) - REF   153.981824    307.963648
2    2040  VRE & ESS (mid) Coal (low) - REF   400.820037    801.640073
3    2050  VRE & ESS (mid) Coal (low) - REF   650.111745   3231.283435


In [87]:
df_ = pd.read_csv(path_to_tables + '/cost-grouped_capacity.csv')
print(df_['technology'].unique())

df_1_ = df_.loc[df_['technology'] == 'Battery'].copy()
df_1_ = df_1_.loc[df_1_['scenario'] == 'VRE & ESS (mid) Coal (low) - REF']


df_1_ = df_1_.groupby(['period', 'scenario']).agg({'capacity_mw': 'sum',
                                                   'capacity_mwh': 'sum'}).reset_index(drop = False)

df_1_['capacity_mw']  /= 1000.
df_1_['capacity_mwh'] /= 1000.
print(df_1_)

df_2_ = df_.loc[df_['technology'] == 'Solar'].copy()
df_2_ = df_2_.loc[df_2_['scenario'] == 'VRE & ESS (mid) Coal (low) - REF']
df_2_ = df_2_.groupby(['period', 
                       'scenario']).agg({'capacity_mw': 'sum',
                                         'capacity_mwh': 'sum'}).reset_index(drop = False)

df_2_['capacity_mw']  /= 1000.
df_2_['capacity_mwh'] /= 1000.
print(df_2_)

df_3_ = df_.loc[df_['technology'] == 'Wind'].copy()
df_3_ = df_3_.loc[df_3_['scenario'] == 'VRE & ESS (mid) Coal (low) - REF']
df_3_ = df_3_.groupby(['period', 
                       'scenario']).agg({'capacity_mw': 'sum',
                                         'capacity_mwh': 'sum'}).reset_index(drop = False)

df_3_['capacity_mw']  /= 1000.
df_3_['capacity_mwh'] /= 1000.
print(df_3_)

df_1_ = df_1_.set_index(["period"])
df_2_ = df_2_.set_index(["period"])
df_3_ = df_3_.set_index(["period"])

print(df_1_.loc[2030, 'capacity_mw']/38.6, df_1_.loc[2030, 'capacity_mwh']/201.5)
print(df_2_.loc[2030, 'capacity_mw'], df_3_.loc[2030, 'capacity_mw'])
print(df_2_.loc[2030, 'capacity_mw'] + df_3_.loc[2030, 'capacity_mw'])

print((df_2_.loc[2030, 'capacity_mw'] - df_2_.loc[2020, 'capacity_mw'])/10.)
print((df_3_.loc[2030, 'capacity_mw'] - df_3_.loc[2020, 'capacity_mw'])/10.)

['Battery' 'Coal' 'Gas' 'Hydro' 'Other' 'Solar' 'Wind' 'Hydrogen' 'Diesel'
 'Nuclear' 'Pumped Storage']
   period                          scenario  capacity_mw  capacity_mwh
0    2020  VRE & ESS (mid) Coal (low) - REF     0.000000      0.000000
1    2030  VRE & ESS (mid) Coal (low) - REF   153.981824    307.963648
2    2040  VRE & ESS (mid) Coal (low) - REF   400.820037    801.640073
3    2050  VRE & ESS (mid) Coal (low) - REF   650.111745   3231.283435
   period                          scenario  capacity_mw  capacity_mwh
0    2020  VRE & ESS (mid) Coal (low) - REF    33.963000           0.0
1    2030  VRE & ESS (mid) Coal (low) - REF   246.365854           0.0
2    2040  VRE & ESS (mid) Coal (low) - REF   995.165281           0.0
3    2050  VRE & ESS (mid) Coal (low) - REF  2138.928134           0.0
   period                          scenario  capacity_mw  capacity_mwh
0    2020  VRE & ESS (mid) Coal (low) - REF    37.715000           0.0
1    2030  VRE & ESS (mid) Coal (low) - REF 

## Cost-optimal deployment of renewable energy and battery storage in 2040 is three times the official targets in 2030

In [135]:
df_ = pd.read_csv(path_to_tables + '/cost-grouped_dispatch.csv')
print(df_['technology'].unique())
print(df_['scenario'].unique())

df_ = df_.groupby(['period', 
                   'technology', 
                   'scenario']).agg({'power_mw': 'sum'}).reset_index(drop = False)

idx_ = ((df_['period'] == 2040) 
        & (df_['technology'] == 'Solar') 
        & (df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'))

df_1_ = df_.loc[idx_].reset_index(drop = True)
print(df_1_)


idx_ = ((df_['period'] == 2040) 
        & (df_['technology'] == 'Wind') 
        & (df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'))

df_2_ = df_.loc[idx_].reset_index(drop = True)
print(df_2_)

idx_ = ((df_['technology'] == 'Battery') 
        & (df_['technology'] == 'Hydrogen')
        & (df_['technology'] == 'Transmission Losses'))

df_3_ = df_.loc[~idx_].reset_index(drop = True)

df_3_ = df_3_.groupby(['period', 
                       'scenario']).agg({'power_mw': 'sum'}).reset_index(drop = False)

idx_ = ((df_3_['period'] == 2040) 
        & (df_3_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'))

df_3_ = df_3_.loc[idx_].reset_index(drop = True)

df_1_ = df_1_.set_index(["period"])
df_2_ = df_2_.set_index(["period"])
df_3_ = df_3_.set_index(["period"])

print(100.*(df_1_.loc[2040, 'power_mw'] + df_2_.loc[2040, 'power_mw'])/df_3_.loc[2040, 'power_mw'])

['Coal' 'Gas' 'Hydro' 'Other' 'Solar' 'Wind' 'Hydrogen' 'Diesel' 'Battery'
 'Nuclear' 'Transmission Losses']
['VREhigh_SThigh_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREhigh_SThigh_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VRElow_STlow_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid']
   period technology                                           scenario  \
0    2040      Solar  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...   

       power_mw  
0  1.655391e+09  
   period technology                                           scenario  \
0    2040       Wind  VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CA...   

       power_mw  
0  1.354283e+09  
70.05379490340727


In [108]:
df_ = pd.read_csv(path_to_tables + '/cost-grouped_capacity.csv')
print(df_['technology'].unique())

df_1_ = df_.loc[df_['technology'] == 'Battery'].copy()
df_1_ = df_1_.loc[df_1_['scenario'] == 'VRE & ESS (mid) Coal (low) - REF']


df_1_ = df_1_.groupby(['period', 'scenario']).agg({'capacity_mw': 'sum',
                                                   'capacity_mwh': 'sum'}).reset_index(drop = False)

df_1_['capacity_mw']  /= 1000.
df_1_['capacity_mwh'] /= 1000.
print(df_1_)

df_2_ = df_.loc[df_['technology'] == 'Solar'].copy()
df_2_ = df_2_.loc[df_2_['scenario'] == 'VRE & ESS (mid) Coal (low) - REF']
df_2_ = df_2_.groupby(['period', 
                       'scenario']).agg({'capacity_mw': 'sum',
                                         'capacity_mwh': 'sum'}).reset_index(drop = False)

df_2_['capacity_mw']  /= 1000.
df_2_['capacity_mwh'] /= 1000.
print(df_2_)

df_3_ = df_.loc[df_['technology'] == 'Wind'].copy()
df_3_ = df_3_.loc[df_3_['scenario'] == 'VRE & ESS (mid) Coal (low) - REF']
df_3_ = df_3_.groupby(['period', 
                       'scenario']).agg({'capacity_mw': 'sum',
                                         'capacity_mwh': 'sum'}).reset_index(drop = False)

df_3_['capacity_mw']  /= 1000.
df_3_['capacity_mwh'] /= 1000.
print(df_3_)

df_1_ = df_1_.set_index(["period"])
df_2_ = df_2_.set_index(["period"])
df_3_ = df_3_.set_index(["period"])

print(df_1_.loc[2040, 'capacity_mw'], df_1_.loc[2040, 'capacity_mwh'])
print(df_2_.loc[2040, 'capacity_mw'], df_3_.loc[2040, 'capacity_mw'])


['Battery' 'Coal' 'Gas' 'Hydro' 'Other' 'Solar' 'Wind' 'Hydrogen' 'Diesel'
 'Nuclear' 'Pumped Storage']
   period                          scenario  capacity_mw  capacity_mwh
0    2020  VRE & ESS (mid) Coal (low) - REF     0.000000      0.000000
1    2030  VRE & ESS (mid) Coal (low) - REF   153.981824    307.963648
2    2040  VRE & ESS (mid) Coal (low) - REF   400.820037    801.640073
3    2050  VRE & ESS (mid) Coal (low) - REF   650.111745   3231.283435
   period                          scenario  capacity_mw  capacity_mwh
0    2020  VRE & ESS (mid) Coal (low) - REF    33.963000           0.0
1    2030  VRE & ESS (mid) Coal (low) - REF   246.365854           0.0
2    2040  VRE & ESS (mid) Coal (low) - REF   995.165281           0.0
3    2050  VRE & ESS (mid) Coal (low) - REF  2138.928134           0.0
   period                          scenario  capacity_mw  capacity_mwh
0    2020  VRE & ESS (mid) Coal (low) - REF    37.715000           0.0
1    2030  VRE & ESS (mid) Coal (low) - REF 

In [148]:
df_ = pd.read_csv(path_to_tables + '/cost-emissions.csv')
print(df_['scenario'].unique())

df_1_ = df_.loc[df_['scenario'] == 'VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid']
df_2_ = df_.loc[df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid']
df_3_ = df_.loc[df_['scenario'] == 'VREhigh_SThigh_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid']

df_1_ = df_1_.set_index(["period"])
df_2_ = df_2_.set_index(["period"])
df_3_ = df_3_.set_index(["period"])

print(1. - df_1_.loc[2040, 'carbon_emissions_tons']/df_1_.loc[2020, 'carbon_emissions_tons'])
print(1. - df_2_.loc[2040, 'carbon_emissions_tons']/df_2_.loc[2020, 'carbon_emissions_tons'])
print(1. - df_3_.loc[2040, 'carbon_emissions_tons']/df_3_.loc[2020, 'carbon_emissions_tons'])

['VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VRElow_STlow_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREhigh_SThigh_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREhigh_SThigh_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid']
0.3203259955107193
0.20429005642099196
0.11869155511946294


In [147]:
df_ = pd.read_csv(path_to_tables + '/pier-emissions.csv')
print(df_['scenario'].unique())

df_1_ = df_.loc[df_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid']

df_ = pd.read_csv(path_to_tables + '/additional-emissions.csv')
print(df_['scenario'].unique())

df_2_ = df_.loc[df_['scenario'] == 'VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid']
# df_3_ = df_.loc[df_['scenario'] == 'VREhigh_SThigh_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid']

df_1_ = df_1_.set_index(["period"])
df_2_ = df_2_.set_index(["period"])
# df_3_ = df_3_.set_index(["period"])

print(1. - df_1_.loc[2040, 'carbon_emissions_tons']/df_1_.loc[2020, 'carbon_emissions_tons'])
print(1. - df_2_.loc[2040, 'carbon_emissions_tons']/df_2_.loc[2020, 'carbon_emissions_tons'])
# print(df_3_.loc[2040, 'carbon_emissions_tons']/df_3_.loc[2020, 'carbon_emissions_tons'])

['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_80CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_100CAP_500GW_PIERmid']
['VREhigh_SThigh_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'
 'VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAPLin_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_50RPS_90CAP_500GW_PIERmid']
0.20432838253828167
0.32148196960442676


## Climate policies or clean energy targets are essential in the long-term to meet ambitious carbon mitigation and net-zero goals

## Electricity demand projections are uncertain but solar-aligned demand can substantially reduce costs

In [133]:
df_ = pd.read_csv(path_to_tables + '/demand-system_cost.csv')
print(df_['scenario'].unique())

df_['total_cost'] = df_['variable_cost'] + df_['fixed_cost'] + 5.9e9

df_p_ = df_.loc[df_['period'] == 2040].reset_index(drop = True).set_index("scenario")

print(df_p_.loc['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_ICEDlow', 'total_cost'] 
      / df_p_.loc['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERlow', 'total_cost'])

print(df_p_.loc['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_ICEDmid', 'total_cost']
      / df_p_.loc['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid', 'total_cost'])

print(df_p_.loc['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_ICEDhigh', 'total_cost']
      / df_p_.loc['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERhigh', 'total_cost'])

df_p_ = df_.loc[df_['period'] == 2050].reset_index(drop = True).set_index("scenario")

print(df_p_.loc['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_ICEDlow', 'total_cost']
      / df_p_.loc['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERlow', 'total_cost'])

print(df_p_.loc['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_ICEDmid', 'total_cost']
      / df_p_.loc['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid', 'total_cost'])

print(df_p_.loc['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_ICEDhigh', 'total_cost']
      / df_p_.loc['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERhigh', 'total_cost'])

['VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_ICEDlow'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_ICEDmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_ICEDhigh'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERlow'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERhigh']
1.069918228215003
1.0679252963394072
1.0682148817196
1.1188298960679464
1.1030326844914697
1.0942100877512715
